<a href="https://colab.research.google.com/github/xyaoaf/UrbanTreeStudywithLidar/blob/main/230306_MachineLearning_Classification_Regression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Import necessary packages
import pandas as pd
import seaborn as sb
import matplotlib.pyplot as plt
import numpy as np
import statsmodels.api as sm
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.linear_model import LogisticRegression, RidgeClassifier
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, recall_score, precision_score, f1_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier 
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.svm import SVC
from sklearn.model_selection import KFold, cross_val_predict, StratifiedKFold
from sklearn.metrics import confusion_matrix, classification_report
from imblearn.over_sampling import RandomOverSampler, SMOTE
from scipy import stats
from scipy.stats import t
from sklearn.linear_model import LinearRegression
# Import the mean squared error function
from sklearn.metrics import mean_squared_error
from sklearn.neighbors import KNeighborsRegressor


# Set pandas display options
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

# Other imports and code omitted


In [ ]:
from sklearn.model_selection import train_test_split
from scipy import stats
from sklearn.metrics import confusion_matrix
from scipy.stats import t
from sklearn.preprocessing import StandardScaler
from scipy.special import stdtr
import pandas as pd
import seaborn as sb
import matplotlib.pyplot as plt
import numpy as np
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, recall_score, precision_score, f1_score
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

In [ ]:
popu = pd.read_csv('/content/drive/MyDrive/XihanYAO_GISDataShare/All_TreeMetrics_Clean.csv')
popuE = pd.read_csv('/content/drive/MyDrive/XihanYAO_GISDataShare/East009701_TreeMetrics_Clean.csv')
popuW = pd.read_csv('/content/drive/MyDrive/XihanYAO_GISDataShare/West002401_TreeMetrics_Clean.csv')
dfTrain = pd.read_csv('/content/drive/MyDrive/XihanYAO_GISDataShare/All_TreeMerge_Clean.csv')
dfTrainE = pd.read_csv('/content/drive/MyDrive/XihanYAO_GISDataShare/East_TreeMerge_Clean.csv')
dfTrainW = pd.read_csv('/content/drive/MyDrive/XihanYAO_GISDataShare/West_TreeMerge_Clean.csv')
population_independent = popu.iloc[:, 5:].astype(np.float64)
population_independent

In [ ]:
dfTrain

In [ ]:
X = dfTrain.iloc[:, 14:].astype(np.float64)
y = dfTrain["DBH"]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=23)

In [ ]:
print(f'X train shape: {X_train.shape}; y train shape: {y_train.shape}')
print(f'X test shape: {X_test.shape}; y test shape: {y_test.shape}')

# Adding new training data of hedge
after adding data, there are more inputs for training, but only with categorial information indicating that those 200 samples are all hedge

In [ ]:
dfhedgeE = pd.read_csv('/content/drive/MyDrive/XihanYAO_GISDataShare/E009701Hedge.csv')
dfhedgeW = pd.read_csv('/content/drive/MyDrive/XihanYAO_GISDataShare/W002401Hedge.csv')
dfhedge = pd.concat([dfhedgeW, dfhedgeE], axis=0)
dfhedge

In [ ]:
dfhedge= dfhedge.drop(['height','winRadius', 'treeIDW', 'crownArea', 'Ground','SArea','Min_Slope','Max_Slope','Max_Slope','Z_Mean','Avg_Slope','treeIDE'], axis=1)

In [ ]:
# merge the two datasets on a common column
dfhedge_popu = pd.merge(dfhedge, popu, on='treeID', how='left')
dfhedge_popu = dfhedge_popu.rename(columns={'OID_': 'OBJECTID'})
dfhedge_popu['Family'] = 'HEDGE'
dfhedge_popu['Functional'] = 'HEDGE'
dfhedge_popu['Genus'] = 'HEDGE'
dfhedge_popu['Common'] = 'HEDGE'
dfhedge_popu['Scientific'] = 'HEDGE'
dfhedge_popu['Condition'] = np.nan
dfhedge_popu['DBH'] = np.nan
dfhedge_popu

In [ ]:
dfTrainHedge = pd.concat([dfTrain, dfhedge_popu], axis=0).reset_index()
dfTrainHedge.loc[:, 'Keep'] = 1
dfTrainHedge['DBH'] = dfTrainHedge['DBH'].fillna(-1)
dfTrainHedge['Condition'] = dfTrainHedge['Condition'].fillna("HEDGE")
dfTrainHedge=dfTrainHedge.dropna()
dfTrainHedge.drop('index', axis=1, inplace=True)
dfTrainHedge

In [ ]:
# Get the set of columns in each dataframe
cols1 = set(dfTrainHedge.columns)
cols2 = set(dfTrain.columns)

# Compare the columns
if cols1 - cols2:
    print("The following columns are in df1 but not in df2:", cols1 - cols2)
if cols2 - cols1:
    print("The following columns are in df2 but not in df1:", cols2 - cols1)
if cols1 == cols2:
    print("The columns in both dataframes are the same.")

In [ ]:
dfTrainHedge

# Linear Regression

In [ ]:
# Import the model we want from sklearn's linear model module
from sklearn.linear_model import LinearRegression

In [ ]:
# Create an object for the model
# We use the default settings
model = LinearRegression()
model

In [ ]:
# Train the model using the fit function
model.fit(X_train, y_train)

In [ ]:
print(f'Training R^2: {model.score(X_train, y_train)}')

In [ ]:
print(f'Test R^2: {model.score(X_test, y_test)}')

In [ ]:
# Obtain predictions
y_train_pred = model.predict(X_train)
y_test_pred = model.predict(X_test)

In [ ]:
print(f'Train RMSE: {mean_squared_error(y_train, y_train_pred, squared=False)}')

In [ ]:
print(f'Test RMSE: {mean_squared_error(y_test, y_test_pred, squared=False)}')

In [ ]:
print(f'Train MSE: {mean_squared_error(y_train, y_train_pred)}')
print(f'Test MSE: {mean_squared_error(y_test, y_test_pred)}')

In [ ]:
y_train_pred = model.predict(X_train)
y_test_pred = model.predict(X_test)

In [ ]:
from sklearn.metrics import mean_absolute_error
print(mean_absolute_error(y_train, y_train_pred))
print(mean_absolute_error(y_test, y_test_pred))

In [ ]:
model.coef_

In [ ]:
X.columns

## K-nearest neighbors regression

In [ ]:
from sklearn.neighbors import KNeighborsRegressor

In [ ]:
def tune_k_neighbors(n_neighbors, X_train, y_train, X_test, y_test):
    # Iterate over the number of neighbors
    for K in n_neighbors:
        # Create nearest neighbors regressors
        knn_reg = KNeighborsRegressor(
            n_neighbors=K,
            weights='uniform',  # ‘distance’ weights points by inverse of their distance
            algorithm='auto',  # out of ‘ball_tree’, ‘kd_tree’, ‘brute’
            leaf_size=30)  # for tree algorithms
        # Fit model
        knn_reg.fit(X_train, y_train)
        # Run predictions
        knn_train_pred = knn_reg.predict(X_train)
        knn_test_pred = knn_reg.predict(X_test)
        # Print summary
        print(f'K={K}: Train RMSE = {mean_squared_error(y_train, knn_train_pred, squared=False):0.4f}; '
              f'Test RMSE: {mean_squared_error(y_test, knn_test_pred, squared=False):0.4f}')


In [ ]:
# Example of hyperparameter tuning for the `k` neighbors value
n_list = [2, 3, 4, 5, 6, 7, 8, 9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25]
tune_k_neighbors(n_list, X_train, y_train, X_test, y_test)

## Regularization - Panelty

In [ ]:
from sklearn.linear_model import Ridge
# Create models
ridge = Ridge(
    # Regularization penalty
    alpha=10,
    random_state=1)
# Fit object
ridge.fit(X_train, y_train)

In [ ]:
# Run predictions
y_train_pred_ridge = ridge.predict(X_train)
y_test_pred_ridge = ridge.predict(X_test)

In [ ]:
# Evaluate model
print(f'Training R^2: {ridge.score(X_train, y_train)}')
print(f'Test R^2: {ridge.score(X_test, y_test)}')
print(f'Train RMSE: {mean_squared_error(y_train, y_train_pred_ridge, squared=False)}')
print(f'Test RMSE: {mean_squared_error(y_test, y_test_pred_ridge, squared=False)}')

In [ ]:
# Linear regression
ols = LinearRegression()
ols.fit(X_train, y_train)
# Ridge, no penalty
ridge2 = Ridge(alpha=0, random_state=2) 
ridge2.fit(X_train, y_train)

In [ ]:
# Evaluate
print(f'Training R^2, Original Ridge: {ridge.score(X_train, y_train)}')
print(f'Test R^2, Original Ridge: {ridge.score(X_test, y_test)}')
print(f'Training R^2, OLS: {ols.score(X_train, y_train)}')
print(f'Test R^2, OLS: {ols.score(X_test, y_test)}')
print(f'Training R^2, Ridge with no penalty: {ridge2.score(X_train, y_train)}')
print(f'Test R^2, Ridge with no penalty: {ridge2.score(X_test, y_test)}')

In [ ]:
from sklearn.linear_model import RidgeCV
# Create ridge model, with CV
ridge_cv = RidgeCV(
    # Which alpha values to test for?
    alphas=np.logspace(-1, 3, 100),
    # Number of folds
    cv=5)
# Fit model
ridge_cv.fit(X_train, y_train)
# Evaluate model
print(ridge_cv.score(X_train, y_train))
print(ridge_cv.score(X_test, y_test))

In [ ]:
ridge_cv.alpha_

In [ ]:
ridge_cv.coef_

In [ ]:
from sklearn.linear_model import Lasso
lasso1 = Lasso(alpha=0.01)
lasso1.fit(X_train, y_train)
lasso1.coef_

In [ ]:
lasso2 = Lasso(alpha=10)
lasso2.fit(X_train, y_train)
lasso2.coef_

In [ ]:
lasso3 = Lasso(alpha=10000)
lasso3.fit(X_train, y_train)
lasso3.coef_

## Classification

In [ ]:
X = dfTrain.iloc[:, 14:].astype(np.float64)
y = dfTrain["Functional"]
y

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

In [ ]:
# Train a decision tree classifier
clf = DecisionTreeClassifier()
clf.fit(X_train, y_train)

In [ ]:
# Make predictions on the test set
y_pred = clf.predict(X_test)

# Calculate the accuracy of the model
accuracy = accuracy_score(y_test, y_pred)
print('Accuracy:', accuracy)
# compare the lists element-wise and store the result in a new list
result = [1 if y_pred[i] == y_test.tolist()[i] else 0 for i in range(len(y_pred))]
# print the result list

# create a DataFrame from the three lists
compare = pd.DataFrame({'col1': result, 'col2': y_pred, 'col3': y_test.tolist()})
pd.set_option('display.max_rows', 10)

compare

# **Bootstrap**

## DBH KNeighborsRegressor

In [ ]:
import seaborn as sns
import scipy.stats as stats
import numpy as np
import matplotlib.pyplot as plt

category = "DBH"
df = dfTrain
df = df.drop(df[df[category] == 0].index)
n_bootstrap = 100
# (467/1260)^2+(260/1260)^2+(231/1260)^2+(85/1260)^2+(78/1260)^2+(75/1260)^2+(64/1260)^2
null_accuracy = 0.2281
clf = LinearRegression(fit_intercept=True, copy_X=True, n_jobs=None, positive=False)
X = df.iloc[:, 14:].astype(np.float64)
y = df[category]
X

model=KNeighborsRegressor(n_neighbors=5, weights='uniform', algorithm='auto', leaf_size=30, p=2, metric='minkowski', metric_params=None, n_jobs=None)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
model.fit(X_train, y_train)
# Make predictions on the test set
y_pred = model.predict(X_test)
# Calculate the accuracy of the model
test_r2 = model.score(X_test, y_test)
test_rmse = mean_squared_error(y_test, y_pred, squared=False)
print("RMSE",test_rmse)



# plot the test_y and pred_y using jointplot with annotations
sns.set_style('whitegrid')
g = sns.jointplot(x=y_test, y=y_pred, kind='scatter', color='steelblue', height=8, ratio=5, marginal_ticks=True)
min_val = min(y_test.min(), y_pred.min())
max_val = max(y_test.max(), y_pred.max())
g.ax_joint.plot([min_val, max_val], [min_val, max_val], color='grey', linestyle='--')
g.ax_joint.set_xlabel('True DBH', fontsize=14)
g.ax_joint.set_ylabel('Predicted DBH', fontsize=14)
# g.ax_joint.set_title('Predicted vs Actual Values', fontsize=16)
plt.show()

In [ ]:
n_bootstrap = 20
# bootstrap
# Bootstrap the data and calculate the test statistic on each bootstrap sample
test_r2 = []
test_RMSE = []

for i in range(n_bootstrap):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
    model.fit(X_train, y_train)
    # Make predictions on the test set
    y_pred = model.predict(X_test)
    # Calculate the accuracy of the model
    test_R2 = model.score(X_test, y_test)
    test_rmse = mean_squared_error(y_test, y_pred, squared=False)
    test_r2.append(test_R2)
    test_RMSE.append(test_rmse)

# Calculate the mean
print("r2 value", np.mean(test_r2))

# Calculate the mean
mean = np.mean(test_RMSE)
# Calculate the standard deviation
std = np.std(test_RMSE)
# Confidence interval
n = len(test_RMSE)
confidence_level = 0.95
ci = t.interval(confidence_level, n - 1, loc=mean, scale=std)
print(f"Confidence interval: [{ci[0]:.3f}, {ci[1]:.3f}]")

# Print the results
print("Mean RMSE: {:.5f}".format(mean))
print("Standard deviation: {:.4f}".format(std))
plt.hist(test_RMSE, density=False)


# DBH Random Forest

In [ ]:
from sklearn.neighbors import RadiusNeighborsRegressor
from sklearn.ensemble import RandomForestRegressor
import seaborn as sns
import scipy.stats as stats
import numpy as np
import matplotlib.pyplot as plt

category = "DBH"
df = dfTrain
df = df.drop(df[df[category] == 0].index)
n_bootstrap = 1
# (467/1260)^2+(260/1260)^2+(231/1260)^2+(85/1260)^2+(78/1260)^2+(75/1260)^2+(64/1260)^2
null_accuracy = 0.2281
model = RandomForestRegressor(n_estimators=100, criterion='squared_error', max_depth=None, min_samples_split=2, min_samples_leaf=1, min_weight_fraction_leaf=0.0, max_features=1.0, max_leaf_nodes=None, min_impurity_decrease=0.0, bootstrap=True, oob_score=False, n_jobs=None, random_state=None, verbose=0, warm_start=False, ccp_alpha=0.0, max_samples=None)
X = df.iloc[:, 14:].astype(np.float64)
y = df[category]
X

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
model.fit(X_train, y_train)
# Make predictions on the test set
y_pred = model.predict(X_test)
# Calculate the accuracy of the model
# test_r2 = model.score(X_test, y_test)
test_rmse = mean_squared_error(y_test, y_pred, squared=False)
print("RMSE",test_rmse)



# plot the test_y and pred_y using jointplot with annotations
sns.set_style('whitegrid')
g = sns.jointplot(x=y_test, y=y_pred, kind='scatter', color='steelblue', height=8, ratio=5, marginal_ticks=True)
min_val = min(y_test.min(), y_pred.min())
max_val = max(y_test.max(), y_pred.max())
g.ax_joint.plot([min_val, max_val], [min_val, max_val], color='grey', linestyle='--')
g.ax_joint.set_xlabel('True DBH', fontsize=14)
g.ax_joint.set_ylabel('Predicted DBH', fontsize=14)
# g.ax_joint.set_title('Predicted vs Actual Values', fontsize=16)
plt.show()

In [ ]:
n_bootstrap = 20
# bootstrap
# Bootstrap the data and calculate the test statistic on each bootstrap sample
test_stats = []
test_RMSE = []

for i in range(n_bootstrap):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
    model.fit(X_train, y_train)
    # Make predictions on the test set
    y_pred = model.predict(X_test)
    # Calculate the accuracy of the model
    test_r2 = model.score(X_test, y_test)
    test_rmse = mean_squared_error(y_test, y_pred, squared=False)
    test_stats.append(test_r2)
    test_RMSE.append(test_rmse)

# Calculate the mean
print("r2 value", np.mean(test_stats))

# Calculate the mean
mean = np.mean(test_RMSE)
# Calculate the standard deviation
std = np.std(test_RMSE)
# Confidence interval
n = len(test_RMSE)
confidence_level = 0.95
ci = t.interval(confidence_level, n - 1, loc=mean, scale=std)
print(f"Confidence interval: [{ci[0]:.3f}, {ci[1]:.3f}]")

# Print the results
print("Mean RMSE: {:.5f}".format(mean))
print("Standard deviation: {:.4f}".format(std))
plt.hist(test_RMSE, density=False)


## DBH Random Forest

In [ ]:
from sklearn.neighbors import RadiusNeighborsRegressor
from sklearn.ensemble import RandomForestRegressor
import seaborn as sns
import scipy.stats as stats
import numpy as np
import matplotlib.pyplot as plt

category = "DBH"
df = dfTrain
df = df.drop(df[df[category] == 0].index)
n_bootstrap = 100
# (467/1260)^2+(260/1260)^2+(231/1260)^2+(85/1260)^2+(78/1260)^2+(75/1260)^2+(64/1260)^2
null_accuracy = 0.2281
model = RandomForestRegressor(n_estimators=100, criterion='squared_error', max_depth=None, min_samples_split=2, min_samples_leaf=1, min_weight_fraction_leaf=0.0, max_features=1.0, max_leaf_nodes=None, min_impurity_decrease=0.0, bootstrap=True, oob_score=False, n_jobs=None, random_state=None, verbose=0, warm_start=False, ccp_alpha=0.0, max_samples=None)
X = df.iloc[:, 14:].astype(np.float64)
y = df[category]
X

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
model.fit(X_train, y_train)
# Make predictions on the test set
y_pred = model.predict(X_test)
# Calculate the accuracy of the model
# test_r2 = model.score(X_test, y_test)
test_rmse = mean_squared_error(y_test, y_pred, squared=False)
print("RMSE",test_rmse)



# plot the test_y and pred_y using jointplot with annotations
sns.set_style('whitegrid')
g = sns.jointplot(x=y_test, y=y_pred, kind='scatter', color='steelblue', height=8, ratio=5, marginal_ticks=True)
min_val = min(y_test.min(), y_pred.min())
max_val = max(y_test.max(), y_pred.max())
g.ax_joint.plot([min_val, max_val], [min_val, max_val], color='grey', linestyle='--')
g.ax_joint.set_xlabel('True DBH', fontsize=14)
g.ax_joint.set_ylabel('Predicted DBH', fontsize=14)
# g.ax_joint.set_title('Predicted vs Actual Values', fontsize=16)
plt.show()

In [ ]:
n_bootstrap = 20
# bootstrap
# Bootstrap the data and calculate the test statistic on each bootstrap sample
test_stats = []
test_RMSE = []

for i in range(n_bootstrap):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
    model.fit(X_train, y_train)
    # Make predictions on the test set
    y_pred = model.predict(X_test)
    # Calculate the accuracy of the model
    test_r2 = model.score(X_test, y_test)
    test_rmse = mean_squared_error(y_test, y_pred, squared=False)
    test_stats.append(test_r2)
    test_RMSE.append(test_rmse)

# Calculate the mean
print("r2 value", np.mean(test_stats))

# Calculate the mean
mean = np.mean(test_RMSE)
# Calculate the standard deviation
std = np.std(test_RMSE)
# Confidence interval
n = len(test_RMSE)
confidence_level = 0.95
ci = t.interval(confidence_level, n - 1, loc=mean, scale=std)
print(f"Confidence interval: [{ci[0]:.3f}, {ci[1]:.3f}]")

# Print the results
print("Mean RMSE: {:.5f}".format(mean))
print("Standard deviation: {:.4f}".format(std))
plt.hist(test_RMSE, density=False)


# DBH LinearRegression


In [ ]:
category = "DBH"
df = dfTrain
df = df.drop(df[df[category] == 0].index)
n_bootstrap = 1
# (467/1260)^2+(260/1260)^2+(231/1260)^2+(85/1260)^2+(78/1260)^2+(75/1260)^2+(64/1260)^2
null_accuracy = 0.2281
clf = LinearRegression(fit_intercept=True, copy_X=True, n_jobs=None, positive=False)
X = df.iloc[:, 14:].astype(np.float64)
y = df[category]
X

In [ ]:
model=LinearRegression(fit_intercept=True, copy_X=True, n_jobs=None, positive=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
model.fit(X_train, y_train)
# Make predictions on the test set
y_pred = model.predict(X_test)
# Calculate the accuracy of the model
test_r2 = model.score(X_test, y_test)
test_rmse = mean_squared_error(y_test, y_pred, squared=False)
print("RMSE",test_rmse)

# plot the test_y and pred_y using jointplot with annotations
sns.set_style('whitegrid')
g = sns.jointplot(x=y_test, y=y_pred, kind='scatter', color='steelblue', height=8, ratio=5, marginal_ticks=True)
min_val = min(y_test.min(), y_pred.min())
max_val = max(y_test.max(), y_pred.max())
g.ax_joint.plot([min_val, max_val], [min_val, max_val], color='grey', linestyle='--')
g.ax_joint.set_xlabel('True DBH', fontsize=14)
g.ax_joint.set_ylabel('Predicted DBH', fontsize=14)
# g.ax_joint.set_title('Predicted vs Actual Values', fontsize=16)

plt.show()

In [ ]:
import seaborn as sns
import scipy.stats as stats

model=LinearRegression(fit_intercept=True, copy_X=True, n_jobs=None, positive=True)
y_log = np.log(y)
X_train, X_test, y_train, y_test = train_test_split(X, y_log, test_size=0.2)
model.fit(X_train, y_train)
# Make predictions on the test set
y_log_pred = model.predict(X_test)
y_pred = np.exp(y_log_pred)
# Calculate the accuracy of the model
test_r2 = model.score(X_test, y_test)
test_rmse = mean_squared_error(np.exp(y_test), y_pred, squared=False)
print("RMSE", test_rmse)

# plot the test_y and pred_y using jointplot with annotations
sns.set_style('whitegrid')
g = sns.jointplot(x=np.exp(y_test), y=y_pred, kind='scatter', color='steelblue', height=8, ratio=5, marginal_ticks=True)
min_val = min(np.exp(y_test).min(), y_pred.min())
max_val = max(np.exp(y_test).max(), y_pred.max())
g.ax_joint.plot([min_val, max_val], [min_val, max_val], color='grey', linestyle='--')
g.ax_joint.set_xlabel('True DBH', fontsize=14)
g.ax_joint.set_ylabel('Predicted DBH', fontsize=14)
# g.ax_joint.set_title('Predicted vs Actual Values', fontsize=16)

plt.show()



In [ ]:
# select the column you want to plot
column_data = df['DBH']

# plot the histogram
plt.hist(column_data, bins=20)  # adjust the number of bins as desired
plt.title('Histogram of {}'.format('DBH'))
plt.xlabel('Values')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# bootstrap
# Bootstrap the data and calculate the test statistic on each bootstrap sample
test_r2 = []
test_RMSE = []

for i in range(n_bootstrap):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
    clf.fit(X_train, y_train)
    # Make predictions on the test set
    y_pred = clf.predict(X_test)
    # Calculate the accuracy of the model
    test_r2 = clf.score(X_test, y_test)
    test_rmse = mean_squared_error(y_test, y_pred, squared=False)
    test_stats.append(test_r2)
    test_RMSE.append(test_rmse)
    

print(np.mean(test_RMSE))

In [ ]:
# Calculate the mean
mean = np.mean(test_stats)
# Calculate the standard deviation
std = np.std(test_stats)
# Confidence interval
n = len(test_stats)
confidence_level = 0.95
ci = t.interval(confidence_level, n - 1, loc=mean, scale=std)
print(f"Confidence interval: [{ci[0]:.3f}, {ci[1]:.3f}]")

# Print the results
print("Mean: {:.5f}".format(mean))
print("Standard deviation: {:.4f}".format(std))
plt.hist(test_stats, density=False)

In [ ]:
# Calculate the mean
mean = np.mean(test_RMSE)
# Calculate the standard deviation
std = np.std(test_RMSE)
# Confidence interval
n = len(test_RMSE)
confidence_level = 0.95
ci = t.interval(confidence_level, n - 1, loc=mean, scale=std)
print(f"Confidence interval: [{ci[0]:.3f}, {ci[1]:.3f}]")

# Print the results
print("Mean: {:.5f}".format(mean))
print("Standard deviation: {:.4f}".format(std))
plt.hist(test_RMSE, density=False)

In [ ]:
# # Define the new value to test
# sample_mean = np.mean(test_stats)
# sample_std = np.std(test_stats, ddof=1)
# sample_size = len(test_stats)

# # Calculate the t-statistic and p-value
# t_stat = (sample_mean - null_accuracy) / (sample_std)
# p_value = 2 * stdtr(sample_size - 1, -np.abs(t_stat))

# # Print the results
# print("t-statistic:", t_stat)
# print("p-value:", p_value)

# Deciduous/coniferous (Functional)

In [ ]:
category = "Functional"
df = dfTrain
n_bootstrap = 2
null_accuracy = 0.88797
clf = DecisionTreeClassifier(criterion='entropy', splitter='random', max_depth=5, min_samples_split=5)
X = df.iloc[:, 14:].astype(np.float64)
y = df[category]
X

In [ ]:
counts = df.groupby(category).size().sort_values(ascending=False)
counts.plot(kind= "bar")
print("length:",len(dfTrain))
print(counts)

In [ ]:
# bootstrap
# Bootstrap the data and calculate the test statistic on each bootstrap sample
test_stats = []
for i in range(n_bootstrap):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
    clf.fit(X_train, y_train)
    # Make predictions on the test set
    y_pred = clf.predict(X_test)
    # Calculate the accuracy of the model
    accuracy = accuracy_score(y_test, y_pred)
    test_stats.append(accuracy)
    
# Calculate the mean
mean = np.mean(test_stats)
# Calculate the standard deviation
std = np.std(test_stats)

In [ ]:
# Confidence interval
n = len(test_stats)
confidence_level = 0.95
ci = t.interval(confidence_level, n - 1, loc=mean, scale=std)
print(f"Confidence interval: [{ci[0]:.3f}, {ci[1]:.3f}]")

# Print the results
print("Mean: {:.5f}".format(mean))
print("Standard deviation: {:.4f}".format(std))
plt.hist(test_stats, density=True)

In [ ]:
# Define the new value to test
sample_mean = np.mean(test_stats)
sample_std = np.std(test_stats, ddof=1)
sample_size = len(test_stats)

# Calculate the t-statistic and p-value
t_stat = (sample_mean - null_accuracy) / (sample_std)
p_value = 2 * stdtr(sample_size - 1, -np.abs(t_stat))

# Print the results
print("t-statistic:", t_stat)
print("p-value:", p_value)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
clf.fit(X_train, y_train)
# Make predictions on the test set
y_pred = clf.predict(X_test)
y_trainpred = clf.predict(X_train)
# Calculate the accuracy of the model
accuracyTrain = accuracy_score(y_train, y_trainpred)
accuracy = accuracy_score(y_test, y_pred)
print(accuracyTrain, "Train Accuracy")
print(accuracy, 'Test Accuracy')

label = list(df[category].unique())
label_t = []
for word in label:
    new_word = word + " True"
    label_t.append(new_word)
label_p = []
for word in label:
    new_word = word + " Pred"
    label_p.append(new_word)

# Compute the confusion matrix
conf_mat = confusion_matrix(y_test, y_pred, labels=label)
# Convert the matrix to a pandas DataFrame for easier manipulation
conf_df = pd.DataFrame(conf_mat, index=label_t, columns=label_p)
# Create a heatmap of the confusion matrix using seaborn
sb.heatmap(conf_df, annot=True, cmap='Blues', fmt='g')
# Generate the classification report
report = classification_report(y_test, y_pred)
print(report)

In [ ]:
from sklearn.model_selection import KFold, cross_val_predict
from sklearn.metrics import confusion_matrix
import pandas as pd

# Define the number of folds
n_folds = 5

# Define the cross-validation method
kf = KFold(n_splits=n_folds, shuffle=True, random_state=42)

# Get the predicted target values using cross_val_predict()
y_pred = cross_val_predict(clf, X, y, cv=kf)

# Get the confusion matrix using confusion_matrix()
cm = confusion_matrix(y, y_pred)

# Print the confusion matrix
print("Confusion matrix:")
print(cm)

# Compute the confusion matrix
conf_mat = confusion_matrix(y, y_pred, labels=label)
# Convert the matrix to a pandas DataFrame for easier manipulation
conf_df = pd.DataFrame(conf_mat, index=label_t, columns=label_p)
# Create a heatmap of the confusion matrix using seaborn
sb.heatmap(conf_df, annot=True, cmap='Blues', fmt='g')
# Generate the classification report
report = classification_report(y, y_pred)
print(report)

# Good/Fair/Poor (Condition)

In [ ]:
category = "Condition"
df = dfTrain
n_bootstrap = 2
null_hypotheis = 0.5585
clf = DecisionTreeClassifier()
X = df.iloc[:, 14:].astype(np.float64)
y = df[category]
X

In [ ]:
counts = df.groupby(category).size().sort_values(ascending=False)
counts.plot(kind= "bar")
print("length:",len(dfTrain))
print(counts)

In [ ]:
# bootstrap
# Bootstrap the data and calculate the test statistic on each bootstrap sample
test_stats = []
for i in range(n_bootstrap):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
    clf.fit(X_train, y_train)
    # Make predictions on the test set
    y_pred = clf.predict(X_test)
    # Calculate the accuracy of the model
    accuracy = accuracy_score(y_test, y_pred)
    test_stats.append(accuracy)
    
# Calculate the mean
mean = np.mean(test_stats)
# Calculate the standard deviation
std = np.std(test_stats)

In [ ]:
# Confidence interval
n = len(test_stats)
confidence_level = 0.95
ci = t.interval(confidence_level, n - 1, loc=mean, scale=std)
print(f"Confidence interval: [{ci[0]:.3f}, {ci[1]:.3f}]")

# Print the results
print("Mean: {:.5f}".format(mean))
print("Standard deviation: {:.4f}".format(std))
plt.hist(test_stats, density=True)

In [ ]:
sample_mean = np.mean(test_stats)
sample_std = np.std(test_stats, ddof=1)
sample_size = len(test_stats)

# Calculate the t-statistic and p-value
t_stat = (sample_mean - null_accuracy) / (sample_std)
p_value = 2 * stdtr(sample_size - 1, -np.abs(t_stat))

# Print the results
print("t-statistic:", t_stat)
print("p-value:", p_value)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
clf.fit(X_train, y_train)
# Make predictions on the test set
y_pred = clf.predict(X_test)
y_trainpred = clf.predict(X_train)
# Calculate the accuracy of the model
accuracyTrain = accuracy_score(y_train, y_trainpred)
accuracy = accuracy_score(y_test, y_pred)
print(accuracyTrain, "Train Accuracy")
print(accuracy, 'Test Accuracy')
label = list(df[category].unique())
label_t = []
for word in label:
    new_word = word + " True"
    label_t.append(new_word)
label_p = []
for word in label:
    new_word = word + " Pred"
    label_p.append(new_word)

# Compute the confusion matrix
conf_mat = confusion_matrix(y_test, y_pred, labels=label)
# Convert the matrix to a pandas DataFrame for easier manipulation
conf_df = pd.DataFrame(conf_mat, index=label_t, columns=label_p)
# Create a heatmap of the confusion matrix using seaborn
sb.heatmap(conf_df, annot=True, cmap='Blues', fmt='g')
# Generate the classification report
report = classification_report(y_test, y_pred)
print(report)

In [ ]:
from sklearn.model_selection import KFold, cross_val_predict
from sklearn.metrics import confusion_matrix
import pandas as pd

# Define the number of folds
n_folds = 5

# Define the cross-validation method
kf = KFold(n_splits=n_folds, shuffle=True, random_state=42)

# Get the predicted target values using cross_val_predict()
y_pred = cross_val_predict(clf, X, y, cv=kf)

# Get the confusion matrix using confusion_matrix()
cm = confusion_matrix(y, y_pred)

# Print the confusion matrix
print("Confusion matrix:")
print(cm)

# Compute the confusion matrix
conf_mat = confusion_matrix(y, y_pred, labels=label)
# Convert the matrix to a pandas DataFrame for easier manipulation
conf_df = pd.DataFrame(conf_mat, index=label_t, columns=label_p)
# Create a heatmap of the confusion matrix using seaborn
sb.heatmap(conf_df, annot=True, cmap='Blues', fmt='g')
# Generate the classification report
report = classification_report(y, y_pred)
print(report)

# Deciduous/coniferous/**Hedge** (Functional)

In [ ]:
category = "Functional"
df = dfTrainHedge
n_bootstrap = 2
# calculated as (1342/1656)^2+(229/1656)^2+(85/1656)^2 = 0.6785
null_accuracy = 0.6772
clf = DecisionTreeClassifier()
X = df.iloc[:, 14:].astype(np.float64)
y = df[category]
X

In [ ]:
counts = df.groupby(category).size().sort_values(ascending=False)
counts.plot(kind= "bar")
print("length:",len(df))
print(counts)

In [ ]:
# bootstrap
# Bootstrap the data and calculate the test statistic on each bootstrap sample
test_stats = []
for i in range(n_bootstrap):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
    clf.fit(X_train, y_train)
    # Make predictions on the test set
    y_pred = clf.predict(X_test)
    # Calculate the accuracy of the model
    accuracy = accuracy_score(y_test, y_pred)
    test_stats.append(accuracy)
    
# Calculate the mean
mean = np.mean(test_stats)
# Calculate the standard deviation
std = np.std(test_stats)

In [ ]:
# Confidence interval
n = len(test_stats)
confidence_level = 0.95
ci = t.interval(confidence_level, n - 1, loc=mean, scale=std)
print(f"Confidence interval: [{ci[0]:.3f}, {ci[1]:.3f}]")

# Print the results
print("Mean: {:.5f}".format(mean))
print("Standard deviation: {:.4f}".format(std))
plt.hist(test_stats, density=True)

In [ ]:
# Define the new value to test
sample_mean = np.mean(test_stats)
sample_std = np.std(test_stats, ddof=1)
sample_size = len(test_stats)

# Calculate the t-statistic and p-value
t_stat = (sample_mean - null_accuracy) / (sample_std)
p_value = 2 * stdtr(sample_size - 1, -np.abs(t_stat))

# Print the results
print("t-statistic:", t_stat)
print("p-value:", p_value)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
clf.fit(X_train, y_train)
# Make predictions on the test set
y_pred = clf.predict(X_test)
y_trainpred = clf.predict(X_train)
# Calculate the accuracy of the model
accuracyTrain = accuracy_score(y_train, y_trainpred)
accuracy = accuracy_score(y_test, y_pred)
print(accuracyTrain, "Train Accuracy")
print(accuracy, 'Test Accuracy')

label = list(df[category].unique())
label_t = []
for word in label:
    new_word = word + " True"
    label_t.append(new_word)
label_p = []
for word in label:
    new_word = word + " Pred"
    label_p.append(new_word)

# Compute the confusion matrix
conf_mat = confusion_matrix(y_test, y_pred, labels=label)
# Convert the matrix to a pandas DataFrame for easier manipulation
conf_df = pd.DataFrame(conf_mat, index=label_t, columns=label_p)
# Create a heatmap of the confusion matrix using seaborn
sb.heatmap(conf_df, annot=True, cmap='Blues', fmt='g')
# Generate the classification report
report = classification_report(y_test, y_pred)
print(report)

In [ ]:
# Define the number of folds
n_folds = 5

# Define the cross-validation method
kf = KFold(n_splits=n_folds, shuffle=True, random_state=42)
scores = cross_val_score(clf, X, y, cv=kf)
print(scores)
# Print the cross-validation scores
print(f"Cross-validation scores: {scores}")
print(f"Mean accuracy: {scores.mean():.3f}")
# Train the final model on the full dataset
clf.fit(X, y)

In [ ]:
from sklearn.model_selection import KFold, cross_val_predict
from sklearn.metrics import confusion_matrix
import pandas as pd

# Define the number of folds
n_folds = 5

# Define the cross-validation method
kf = KFold(n_splits=n_folds, shuffle=True, random_state=42)

# Get the predicted target values using cross_val_predict()
y_pred = cross_val_predict(clf, X, y, cv=kf)

# Get the confusion matrix using confusion_matrix()
cm = confusion_matrix(y, y_pred)

# Print the confusion matrix
print("Confusion matrix:")
print(cm)

# Compute the confusion matrix
conf_mat = confusion_matrix(y, y_pred, labels=label)
# Convert the matrix to a pandas DataFrame for easier manipulation
conf_df = pd.DataFrame(conf_mat, index=label_t, columns=label_p)
# Create a heatmap of the confusion matrix using seaborn
sb.heatmap(conf_df, annot=True, cmap='Blues', fmt='g')
# Generate the classification report
report = classification_report(y, y_pred)
print(report)

# 6 Families and **Hedge** (Family)(DecisionTreeClassifier)(57.49%)

---



In [ ]:
category = "Family"
df = dfTrainHedge[dfTrainHedge['Family'].isin(['Sapindaceae', 'Rosaceae', 'Fagaceae','Cornaceae','Betulaceae','Pinaceae','HEDGE'])]
n_bootstrap = 2
# (467/1260)^2+(260/1260)^2+(231/1260)^2+(85/1260)^2+(78/1260)^2+(75/1260)^2+(64/1260)^2
null_accuracy = 0.2281
clf = DecisionTreeClassifier(criterion='gini', splitter='best', max_depth=8, min_samples_split=3)
# clf = DecisionTreeClassifier(criterion='entropy', splitter='random', max_depth=5, min_samples_split=5)
# clf = DecisionTreeClassifier(criterion='entropy', splitter='best', max_depth=8, min_samples_split=3)
X = df.iloc[:, 14:].astype(np.float64)
y = df[category]
X

In [ ]:
counts = df.groupby(category).size().sort_values(ascending=False)
counts.plot(kind= "bar")
print("length:",len(dfTrain))
print(counts)

In [ ]:
# bootstrap
# Bootstrap the data and calculate the test statistic on each bootstrap sample
test_stats = []
for i in range(n_bootstrap):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
    clf.fit(X_train, y_train)
    # Make predictions on the test set
    y_pred = clf.predict(X_test)
    # Calculate the accuracy of the model
    accuracy = accuracy_score(y_test, y_pred)
    test_stats.append(accuracy)
    
# Calculate the mean
mean = np.mean(test_stats)
# Calculate the standard deviation
std = np.std(test_stats)

In [ ]:
# Confidence interval
n = len(test_stats)
confidence_level = 0.95
ci = t.interval(confidence_level, n - 1, loc=mean, scale=std)
print(f"Confidence interval: [{ci[0]:.3f}, {ci[1]:.3f}]")

# Print the results
print("Mean: {:.5f}".format(mean))
print("Standard deviation: {:.4f}".format(std))
plt.hist(test_stats, density=True)

In [ ]:
# Define the new value to test
sample_mean = np.mean(test_stats)
sample_std = np.std(test_stats, ddof=1)
sample_size = len(test_stats)

# Calculate the t-statistic and p-value
t_stat = (sample_mean - null_accuracy) / (sample_std)
p_value = 2 * stdtr(sample_size - 1, -np.abs(t_stat))

# Print the results
print("t-statistic:", t_stat)
print("p-value:", p_value)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
clf.fit(X_train, y_train)
# Make predictions on the test set
y_pred = clf.predict(X_test)
y_trainpred = clf.predict(X_train)
# Calculate the accuracy of the model
accuracyTrain = accuracy_score(y_train, y_trainpred)
accuracy = accuracy_score(y_test, y_pred)
print(accuracyTrain, "Train Accuracy")
print(accuracy, 'Test Accuracy')

label = list(df[category].unique())
label_t = []
for word in label:
    new_word = word + " True"
    label_t.append(new_word)
label_p = []
for word in label:
    new_word = word + " Pred"
    label_p.append(new_word)

# Compute the confusion matrix
conf_mat = confusion_matrix(y_test, y_pred, labels=label)
# Convert the matrix to a pandas DataFrame for easier manipulation
conf_df = pd.DataFrame(conf_mat, index=label_t, columns=label_p)
# Create a heatmap of the confusion matrix using seaborn
sb.heatmap(conf_df, annot=True, cmap='Blues', fmt='g')
# Generate the classification report
report = classification_report(y_test, y_pred)
print(report)

In [ ]:
# Define the number of folds
n_folds = 5

# Define the cross-validation method
kf = KFold(n_splits=n_folds, shuffle=True, random_state=42)
scores = cross_val_score(clf, X, y, cv=kf)
print(scores)
# Print the cross-validation scores
print(f"Cross-validation scores: {scores}")
print(f"Mean accuracy: {scores.mean():.3f}")
# Train the final model on the full dataset
clf.fit(X, y)

In [ ]:
from sklearn.model_selection import KFold, cross_val_predict
from sklearn.metrics import confusion_matrix
import pandas as pd

# Define the number of folds
n_folds = 5

# Define the cross-validation method
kf = KFold(n_splits=n_folds, shuffle=True, random_state=42)

# Get the predicted target values using cross_val_predict()
y_pred = cross_val_predict(clf, X, y, cv=kf)

# Get the confusion matrix using confusion_matrix()
cm = confusion_matrix(y, y_pred)

# Print the confusion matrix
print("Confusion matrix:")
print(cm)

# Compute the confusion matrix
conf_mat = confusion_matrix(y, y_pred, labels=label)
# Convert the matrix to a pandas DataFrame for easier manipulation
conf_df = pd.DataFrame(conf_mat, index=label_t, columns=label_p)
# Create a heatmap of the confusion matrix using seaborn
sb.heatmap(conf_df, annot=True, cmap='Blues', fmt='g')
# Generate the classification report
report = classification_report(y, y_pred)
print(report)

# 6 Families and **Hedge** (Family)(RidgeClassifier)(64.43%)

In [ ]:
category = "Family"
df = dfTrainHedge[dfTrainHedge['Family'].isin(['Sapindaceae', 'Rosaceae', 'Fagaceae','Cornaceae','Betulaceae','Pinaceae','HEDGE'])]
n_bootstrap = 2
# (467/1260)^2+(260/1260)^2+(231/1260)^2+(85/1260)^2+(78/1260)^2+(75/1260)^2+(64/1260)^2
null_accuracy = 0.2281
clf = RidgeClassifier(alpha=1.0)
X = df.iloc[:, 14:].astype(np.float64)
y = df[category]
X

In [ ]:
counts = df.groupby(category).size().sort_values(ascending=False)
counts.plot(kind= "bar")
print("length:",len(dfTrain))
print(counts)

In [ ]:
# bootstrap
# Bootstrap the data and calculate the test statistic on each bootstrap sample
test_stats = []
for i in range(n_bootstrap):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
    clf.fit(X_train, y_train)
    # Make predictions on the test set
    y_pred = clf.predict(X_test)
    # Calculate the accuracy of the model
    accuracy = accuracy_score(y_test, y_pred)
    test_stats.append(accuracy)
    
# Calculate the mean
mean = np.mean(test_stats)
# Calculate the standard deviation
std = np.std(test_stats)

In [ ]:
# Confidence interval
n = len(test_stats)
confidence_level = 0.95
ci = t.interval(confidence_level, n - 1, loc=mean, scale=std)
print(f"Confidence interval: [{ci[0]:.3f}, {ci[1]:.3f}]")

# Print the results
print("Mean: {:.5f}".format(mean))
print("Standard deviation: {:.4f}".format(std))
plt.hist(test_stats, density=True)

In [ ]:
# Define the new value to test
sample_mean = np.mean(test_stats)
sample_std = np.std(test_stats, ddof=1)
sample_size = len(test_stats)

# Calculate the t-statistic and p-value
t_stat = (sample_mean - null_accuracy) / (sample_std)
p_value = 2 * stdtr(sample_size - 1, -np.abs(t_stat))

# Print the results
print("t-statistic:", t_stat)
print("p-value:", p_value)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
clf.fit(X_train, y_train)
# Make predictions on the test set
y_pred = clf.predict(X_test)
y_trainpred = clf.predict(X_train)
# Calculate the accuracy of the model
accuracyTrain = accuracy_score(y_train, y_trainpred)
accuracy = accuracy_score(y_test, y_pred)
print(accuracyTrain, "Train Accuracy")
print(accuracy, 'Test Accuracy')

label = list(df[category].unique())
label_t = []
for word in label:
    new_word = word + " True"
    label_t.append(new_word)
label_p = []
for word in label:
    new_word = word + " Pred"
    label_p.append(new_word)

# Compute the confusion matrix
conf_mat = confusion_matrix(y_test, y_pred, labels=label)
# Convert the matrix to a pandas DataFrame for easier manipulation
conf_df = pd.DataFrame(conf_mat, index=label_t, columns=label_p)
# Create a heatmap of the confusion matrix using seaborn
sb.heatmap(conf_df, annot=True, cmap='Blues', fmt='g')
# Generate the classification report
report = classification_report(y_test, y_pred)
print(report)

In [ ]:
# Define the number of folds
n_folds = 5

# Define the cross-validation method
kf = KFold(n_splits=n_folds, shuffle=True, random_state=42)
scores = cross_val_score(clf, X, y, cv=kf)
print(scores)
# Print the cross-validation scores
print(f"Cross-validation scores: {scores}")
print(f"Mean accuracy: {scores.mean():.3f}")
# Train the final model on the full dataset
clf.fit(X, y)

In [ ]:
from sklearn.model_selection import KFold, cross_val_predict
from sklearn.metrics import confusion_matrix
import pandas as pd

# Define the number of folds
n_folds = 5

# Define the cross-validation method
kf = KFold(n_splits=n_folds, shuffle=True, random_state=42)

# Get the predicted target values using cross_val_predict()
y_pred = cross_val_predict(clf, X, y, cv=kf)

# Get the confusion matrix using confusion_matrix()
cm = confusion_matrix(y, y_pred)

# Print the confusion matrix
print("Confusion matrix:")
print(cm)

# Compute the confusion matrix
conf_mat = confusion_matrix(y, y_pred, labels=label)
# Convert the matrix to a pandas DataFrame for easier manipulation
conf_df = pd.DataFrame(conf_mat, index=label_t, columns=label_p)
# Create a heatmap of the confusion matrix using seaborn
sb.heatmap(conf_df, annot=True, cmap='Blues', fmt='g')
# Generate the classification report
report = classification_report(y, y_pred)
print(report)

In [ ]:
category = "Family"
df = dfTrainHedge[dfTrainHedge['Family'].isin(['Sapindaceae', 'Rosaceae', 'Fagaceae','Cornaceae','Betulaceae','Pinaceae','HEDGE'])]
n_bootstrap = 100
# (467/1260)^2+(260/1260)^2+(231/1260)^2+(85/1260)^2+(78/1260)^2+(75/1260)^2+(64/1260)^2
null_accuracy = 0.2281
clf = LinearDiscriminantAnalysis(solver='lsqr', shrinkage=None, priors=None, n_components=None, store_covariance=False, tol=0.0001, covariance_estimator=None)
X = df.iloc[:, 14:].astype(np.float64)
y = df[category]
X

In [ ]:
counts = df.groupby(category).size().sort_values(ascending=False)
counts.plot(kind= "bar")
print("length:",len(dfTrain))
print(counts)

In [ ]:
# bootstrap
# Bootstrap the data and calculate the test statistic on each bootstrap sample
test_stats = []
for i in range(n_bootstrap):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
    clf.fit(X_train, y_train)
    # Make predictions on the test set
    y_pred = clf.predict(X_test)
    # Calculate the accuracy of the model
    accuracy = accuracy_score(y_test, y_pred)
    test_stats.append(accuracy)
    
# Calculate the mean
mean = np.mean(test_stats)
# Calculate the standard deviation
std = np.std(test_stats)

In [ ]:
# Confidence interval
n = len(test_stats)
confidence_level = 0.95
ci = t.interval(confidence_level, n - 1, loc=mean, scale=std)
print(f"Confidence interval: [{ci[0]:.3f}, {ci[1]:.3f}]")

# Print the results
print("Mean: {:.5f}".format(mean))
print("Standard deviation: {:.4f}".format(std))
plt.hist(test_stats, density=True)

In [ ]:
# Define the new value to test
sample_mean = np.mean(test_stats)
sample_std = np.std(test_stats, ddof=1)
sample_size = len(test_stats)

# Calculate the t-statistic and p-value
t_stat = (sample_mean - null_accuracy) / (sample_std)
p_value = 2 * stdtr(sample_size - 1, -np.abs(t_stat))

# Print the results
print("t-statistic:", t_stat)
print("p-value:", p_value)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
clf.fit(X_train, y_train)
# Make predictions on the test set
y_pred = clf.predict(X_test)
y_trainpred = clf.predict(X_train)
# Calculate the accuracy of the model
accuracyTrain = accuracy_score(y_train, y_trainpred)
accuracy = accuracy_score(y_test, y_pred)
print(accuracyTrain, "Train Accuracy")
print(accuracy, 'Test Accuracy')

label = list(df[category].unique())
label_t = []
for word in label:
    new_word = word + " True"
    label_t.append(new_word)
label_p = []
for word in label:
    new_word = word + " Pred"
    label_p.append(new_word)

# Compute the confusion matrix
conf_mat = confusion_matrix(y_test, y_pred, labels=label)
# Convert the matrix to a pandas DataFrame for easier manipulation
conf_df = pd.DataFrame(conf_mat, index=label_t, columns=label_p)
# Create a heatmap of the confusion matrix using seaborn
sb.heatmap(conf_df, annot=True, cmap='Blues', fmt='g')
# Generate the classification report
report = classification_report(y_test, y_pred)
print(report)

In [ ]:
# Define the number of folds
n_folds = 5

# Define the cross-validation method
kf = KFold(n_splits=n_folds, shuffle=True, random_state=42)
scores = cross_val_score(clf, X, y, cv=kf)
print(scores)
# Print the cross-validation scores
print(f"Cross-validation scores: {scores}")
print(f"Mean accuracy: {scores.mean():.3f}")
# Train the final model on the full dataset
clf.fit(X, y)

In [ ]:
from sklearn.model_selection import KFold, cross_val_predict
from sklearn.metrics import confusion_matrix
import pandas as pd

# Define the number of folds
n_folds = 5

# Define the cross-validation method
kf = KFold(n_splits=n_folds, shuffle=True, random_state=42)

# Get the predicted target values using cross_val_predict()
y_pred = cross_val_predict(clf, X, y, cv=kf)

# Get the confusion matrix using confusion_matrix()
cm = confusion_matrix(y, y_pred)

# Print the confusion matrix
print("Confusion matrix:")
print(cm)

# Compute the confusion matrix
conf_mat = confusion_matrix(y, y_pred, labels=label)
# Convert the matrix to a pandas DataFrame for easier manipulation
conf_df = pd.DataFrame(conf_mat, index=label_t, columns=label_p)
# Create a heatmap of the confusion matrix using seaborn
sb.heatmap(conf_df, annot=True, cmap='Blues', fmt='g')
# Generate the classification report
report = classification_report(y, y_pred)
print(report)

In [ ]:
# LDA contribution test: 
coef = clf.coef_
# for i, col in enumerate (X_train.columns):
#     print (f"{col}: {coef[0, i]:.3f}")

# 6 Families and **Hedge** (Family)
LinearDiscriminantAnalysis
svd:66.81%
lsqr: 67.25% 


In [ ]:
category = "Family"
df = dfTrainHedge[dfTrainHedge['Family'].isin(['Sapindaceae', 'Rosaceae', 'Fagaceae','Cornaceae','Betulaceae','Pinaceae','HEDGE'])]
n_bootstrap = 100
# (467/1260)^2+(260/1260)^2+(231/1260)^2+(85/1260)^2+(78/1260)^2+(75/1260)^2+(64/1260)^2
null_accuracy = 0.2281
clf = LinearDiscriminantAnalysis(solver='lsqr', shrinkage=None, priors=None, n_components=None, store_covariance=False, tol=0.0001, covariance_estimator=None)
X = df.iloc[:, 14:].astype(np.float64)
y = df[category]
X

In [ ]:
counts = df.groupby(category).size().sort_values(ascending=False)
counts.plot(kind= "bar")
print("length:",len(dfTrain))
print(counts)

In [ ]:
# bootstrap
# Bootstrap the data and calculate the test statistic on each bootstrap sample
test_stats = []
for i in range(n_bootstrap):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
    clf.fit(X_train, y_train)
    # Make predictions on the test set
    y_pred = clf.predict(X_test)
    # Calculate the accuracy of the model
    accuracy = accuracy_score(y_test, y_pred)
    test_stats.append(accuracy)
    
# Calculate the mean
mean = np.mean(test_stats)
# Calculate the standard deviation
std = np.std(test_stats)

In [ ]:
# Confidence interval
n = len(test_stats)
confidence_level = 0.95
ci = t.interval(confidence_level, n - 1, loc=mean, scale=std)
print(f"Confidence interval: [{ci[0]:.3f}, {ci[1]:.3f}]")

# Print the results
print("Mean: {:.5f}".format(mean))
print("Standard deviation: {:.4f}".format(std))
plt.hist(test_stats, density=True)

In [ ]:
# Define the new value to test
sample_mean = np.mean(test_stats)
sample_std = np.std(test_stats, ddof=1)
sample_size = len(test_stats)

# Calculate the t-statistic and p-value
t_stat = (sample_mean - null_accuracy) / (sample_std)
p_value = 2 * stdtr(sample_size - 1, -np.abs(t_stat))

# Print the results
print("t-statistic:", t_stat)
print("p-value:", p_value)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
clf.fit(X_train, y_train)
# Make predictions on the test set
y_pred = clf.predict(X_test)
y_trainpred = clf.predict(X_train)
# Calculate the accuracy of the model
accuracyTrain = accuracy_score(y_train, y_trainpred)
accuracy = accuracy_score(y_test, y_pred)
print(accuracyTrain, "Train Accuracy")
print(accuracy, 'Test Accuracy')

label = list(df[category].unique())
label_t = []
for word in label:
    new_word = word + " True"
    label_t.append(new_word)
label_p = []
for word in label:
    new_word = word + " Pred"
    label_p.append(new_word)

# Compute the confusion matrix
conf_mat = confusion_matrix(y_test, y_pred, labels=label)
# Convert the matrix to a pandas DataFrame for easier manipulation
conf_df = pd.DataFrame(conf_mat, index=label_t, columns=label_p)
# Create a heatmap of the confusion matrix using seaborn
sb.heatmap(conf_df, annot=True, cmap='Blues', fmt='g')
# Generate the classification report
report = classification_report(y_test, y_pred)
print(report)

In [ ]:
# Define the number of folds
n_folds = 5

# Define the cross-validation method
kf = KFold(n_splits=n_folds, shuffle=True, random_state=42)
scores = cross_val_score(clf, X, y, cv=kf)
print(scores)
# Print the cross-validation scores
print(f"Cross-validation scores: {scores}")
print(f"Mean accuracy: {scores.mean():.3f}")
# Train the final model on the full dataset
clf.fit(X, y)

In [ ]:
from sklearn.model_selection import KFold, cross_val_predict
from sklearn.metrics import confusion_matrix
import pandas as pd

# Define the number of folds
n_folds = 5

# Define the cross-validation method
kf = KFold(n_splits=n_folds, shuffle=True, random_state=42)

# Get the predicted target values using cross_val_predict()
y_pred = cross_val_predict(clf, X, y, cv=kf)

# Get the confusion matrix using confusion_matrix()
cm = confusion_matrix(y, y_pred)

# Print the confusion matrix
print("Confusion matrix:")
print(cm)

# Compute the confusion matrix
conf_mat = confusion_matrix(y, y_pred, labels=label)
# Convert the matrix to a pandas DataFrame for easier manipulation
conf_df = pd.DataFrame(conf_mat, index=label_t, columns=label_p)
# Create a heatmap of the confusion matrix using seaborn
sb.heatmap(conf_df, annot=True, cmap='Blues', fmt='g')
# Generate the classification report
report = classification_report(y, y_pred)
print(report)

In [ ]:
# LDA contribution test: 
coef = clf.coef_
# for i, col in enumerate (X_train.columns):
#     print (f"{col}: {coef[0, i]:.3f}")

# 6 Families and **Hedge** (Family)
LogisticRegression

In [ ]:
category = "Family"
df = dfTrainHedge[dfTrainHedge['Family'].isin(['Sapindaceae', 'Rosaceae', 'Fagaceae','Cornaceae','Betulaceae','Pinaceae','HEDGE'])]
n_bootstrap = 2
# (467/1260)^2+(260/1260)^2+(231/1260)^2+(85/1260)^2+(78/1260)^2+(75/1260)^2+(64/1260)^2
null_accuracy = 0.2281
clf = LogisticRegression()
X = df.iloc[:, 14:].astype(np.float64)
y = df[category]
X

In [ ]:
counts = df.groupby(category).size().sort_values(ascending=False)
counts.plot(kind= "bar")
print("length:",len(dfTrain))
print(counts)

In [ ]:
# bootstrap
# Bootstrap the data and calculate the test statistic on each bootstrap sample
test_stats = []
for i in range(n_bootstrap):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
    clf.fit(X_train, y_train)
    # Make predictions on the test set
    y_pred = clf.predict(X_test)
    # Calculate the accuracy of the model
    accuracy = accuracy_score(y_test, y_pred)
    test_stats.append(accuracy)
    
# Calculate the mean
mean = np.mean(test_stats)
# Calculate the standard deviation
std = np.std(test_stats)

In [ ]:
# Confidence interval
n = len(test_stats)
confidence_level = 0.95
ci = t.interval(confidence_level, n - 1, loc=mean, scale=std)
print(f"Confidence interval: [{ci[0]:.3f}, {ci[1]:.3f}]")

# Print the results
print("Mean: {:.5f}".format(mean))
print("Standard deviation: {:.4f}".format(std))
plt.hist(test_stats, density=True)

In [ ]:
# Define the new value to test
sample_mean = np.mean(test_stats)
sample_std = np.std(test_stats, ddof=1)
sample_size = len(test_stats)

# Calculate the t-statistic and p-value
t_stat = (sample_mean - null_accuracy) / (sample_std)
p_value = 2 * stdtr(sample_size - 1, -np.abs(t_stat))

# Print the results
print("t-statistic:", t_stat)
print("p-value:", p_value)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
clf.fit(X_train, y_train)
# Make predictions on the test set
y_pred = clf.predict(X_test)
y_trainpred = clf.predict(X_train)
# Calculate the accuracy of the model
accuracyTrain = accuracy_score(y_train, y_trainpred)
accuracy = accuracy_score(y_test, y_pred)
print(accuracyTrain, "Train Accuracy")
print(accuracy, 'Test Accuracy')

label = list(df[category].unique())
label_t = []
for word in label:
    new_word = word + " True"
    label_t.append(new_word)
label_p = []
for word in label:
    new_word = word + " Pred"
    label_p.append(new_word)

# Compute the confusion matrix
conf_mat = confusion_matrix(y_test, y_pred, labels=label)
# Convert the matrix to a pandas DataFrame for easier manipulation
conf_df = pd.DataFrame(conf_mat, index=label_t, columns=label_p)
# Create a heatmap of the confusion matrix using seaborn
sb.heatmap(conf_df, annot=True, cmap='Blues', fmt='g')
# Generate the classification report
report = classification_report(y_test, y_pred)
print(report)

In [ ]:
# Define the number of folds
n_folds = 5

# Define the cross-validation method
kf = KFold(n_splits=n_folds, shuffle=True, random_state=42)
scores = cross_val_score(clf, X, y, cv=kf)
print(scores)
# Print the cross-validation scores
print(f"Cross-validation scores: {scores}")
print(f"Mean accuracy: {scores.mean():.3f}")
# Train the final model on the full dataset
clf.fit(X, y)

In [ ]:
from sklearn.model_selection import KFold, cross_val_predict
from sklearn.metrics import confusion_matrix
import pandas as pd

# Define the number of folds
n_folds = 5

# Define the cross-validation method
kf = KFold(n_splits=n_folds, shuffle=True, random_state=42)

# Get the predicted target values using cross_val_predict()
y_pred = cross_val_predict(clf, X, y, cv=kf)

# Get the confusion matrix using confusion_matrix()
cm = confusion_matrix(y, y_pred)

# Print the confusion matrix
print("Confusion matrix:")
print(cm)

# Compute the confusion matrix
conf_mat = confusion_matrix(y, y_pred, labels=label)
# Convert the matrix to a pandas DataFrame for easier manipulation
conf_df = pd.DataFrame(conf_mat, index=label_t, columns=label_p)
# Create a heatmap of the confusion matrix using seaborn
sb.heatmap(conf_df, annot=True, cmap='Blues', fmt='g')
# Generate the classification report
report = classification_report(y, y_pred)
print(report)

In [ ]:
# LDA contribution test: 
coef = clf.coef_
# for i, col in enumerate (X_train.columns):
#     print (f"{col}: {coef[0, i]:.3f}")

# 6 Families and **Hedge** (Family)
RandomForestClassifier
Default 68.24%

In [ ]:
category = "Family"
df = dfTrainHedge[dfTrainHedge['Family'].isin(['Sapindaceae', 'Rosaceae', 'Fagaceae','Cornaceae','Betulaceae','Pinaceae','HEDGE'])]
n_bootstrap = 2
# (467/1260)^2+(260/1260)^2+(231/1260)^2+(85/1260)^2+(78/1260)^2+(75/1260)^2+(64/1260)^2
null_accuracy = 0.2281
clf = RandomForestClassifier(n_estimators=200, criterion='gini')
X = df.iloc[:, 14:].astype(np.float64)
y = df[category]
X

In [ ]:
counts = df.groupby(category).size().sort_values(ascending=False)
counts.plot(kind= "bar")
print("length:",len(dfTrain))
print(counts)

In [ ]:
# bootstrap
# Bootstrap the data and calculate the test statistic on each bootstrap sample
test_stats = []
for i in range(n_bootstrap):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
    clf.fit(X_train, y_train)
    # Make predictions on the test set
    y_pred = clf.predict(X_test)
    # Calculate the accuracy of the model
    accuracy = accuracy_score(y_test, y_pred)
    test_stats.append(accuracy)
    
# Calculate the mean
mean = np.mean(test_stats)
# Calculate the standard deviation
std = np.std(test_stats)

In [ ]:
# Confidence interval
n = len(test_stats)
confidence_level = 0.95
ci = t.interval(confidence_level, n - 1, loc=mean, scale=std)
print(f"Confidence interval: [{ci[0]:.3f}, {ci[1]:.3f}]")

# Print the results
print("Mean: {:.5f}".format(mean))
print("Standard deviation: {:.4f}".format(std))
plt.hist(test_stats, density=True)

In [ ]:
# Define the new value to test
sample_mean = np.mean(test_stats)
sample_std = np.std(test_stats, ddof=1)
sample_size = len(test_stats)

# Calculate the t-statistic and p-value
t_stat = (sample_mean - null_accuracy) / (sample_std)
p_value = 2 * stdtr(sample_size - 1, -np.abs(t_stat))

# Print the results
print("t-statistic:", t_stat)
print("p-value:", p_value)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
clf.fit(X_train, y_train)
# Make predictions on the test set
y_pred = clf.predict(X_test)
y_trainpred = clf.predict(X_train)
# Calculate the accuracy of the model
accuracyTrain = accuracy_score(y_train, y_trainpred)
accuracy = accuracy_score(y_test, y_pred)
print(accuracyTrain, "Train Accuracy")
print(accuracy, 'Test Accuracy')

label = list(df[category].unique())
label_t = []
for word in label:
    new_word = word + " True"
    label_t.append(new_word)
label_p = []
for word in label:
    new_word = word + " Pred"
    label_p.append(new_word)

# Compute the confusion matrix
conf_mat = confusion_matrix(y_test, y_pred, labels=label)
# Convert the matrix to a pandas DataFrame for easier manipulation
conf_df = pd.DataFrame(conf_mat, index=label_t, columns=label_p)
# Create a heatmap of the confusion matrix using seaborn
sb.heatmap(conf_df, annot=True, cmap='Blues', fmt='g')
# Generate the classification report
report = classification_report(y_test, y_pred)
print(report)

In [ ]:
# Define the number of folds
n_folds = 5

# Define the cross-validation method
kf = KFold(n_splits=n_folds, shuffle=True, random_state=42)
scores = cross_val_score(clf, X, y, cv=kf)
print(scores)
# Print the cross-validation scores
print(f"Cross-validation scores: {scores}")
print(f"Mean accuracy: {scores.mean():.3f}")
# Train the final model on the full dataset
clf.fit(X, y)

In [ ]:
from sklearn.model_selection import KFold, cross_val_predict
from sklearn.metrics import confusion_matrix
import pandas as pd

# Define the number of folds
n_folds = 5

# Define the cross-validation method
kf = KFold(n_splits=n_folds, shuffle=True, random_state=42)

# Get the predicted target values using cross_val_predict()
y_pred = cross_val_predict(clf, X, y, cv=kf)

# Get the confusion matrix using confusion_matrix()
cm = confusion_matrix(y, y_pred)

# Print the confusion matrix
print("Confusion matrix:")
print(cm)

# Compute the confusion matrix
conf_mat = confusion_matrix(y, y_pred, labels=label)
# Convert the matrix to a pandas DataFrame for easier manipulation
conf_df = pd.DataFrame(conf_mat, index=label_t, columns=label_p)
# Create a heatmap of the confusion matrix using seaborn
sb.heatmap(conf_df, annot=True, cmap='Blues', fmt='g')
# Generate the classification report
report = classification_report(y, y_pred)
print(report)

In [ ]:
# Get the feature importance scores
importance_scores = clf.feature_importances_
feature_names = df.iloc[:, 15:].columns[:-1]
# Print the feature importance scores
for feature_name, importance_score in zip(feature_names, importance_scores):
    print(f'{feature_name}: {importance_score:.4f}')

# 6 Families and **Hedge** (Family)(MLPClassifier)(27%)(BAD)

In [ ]:
category = "Family"
df = dfTrainHedge[dfTrainHedge['Family'].isin(['Sapindaceae', 'Rosaceae', 'Fagaceae','Cornaceae','Betulaceae','Pinaceae','HEDGE'])]
n_bootstrap = 1
# (467/1260)^2+(260/1260)^2+(231/1260)^2+(85/1260)^2+(78/1260)^2+(75/1260)^2+(64/1260)^2
null_accuracy = 0.2281
clf = MLPClassifier(hidden_layer_sizes=(100,), activation='relu', solver='adam', alpha=0.0001, batch_size='auto', learning_rate='constant', learning_rate_init=0.001, power_t=0.5, max_iter=200, shuffle=True, random_state=None, tol=0.0001, verbose=False, warm_start=False, momentum=0.9, nesterovs_momentum=True, early_stopping=False, validation_fraction=0.1, beta_1=0.9, beta_2=0.999, epsilon=1e-08, n_iter_no_change=10, max_fun=15000)
X = df.iloc[:, 14:].astype(np.float64)
y = df[category]
X

In [ ]:
counts = df.groupby(category).size().sort_values(ascending=False)
counts.plot(kind= "bar")
print("length:",len(dfTrain))
print(counts)

In [ ]:
# bootstrap
# Bootstrap the data and calculate the test statistic on each bootstrap sample
test_stats = []
for i in range(n_bootstrap):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
    clf.fit(X_train, y_train)
    # Make predictions on the test set
    y_pred = clf.predict(X_test)
    # Calculate the accuracy of the model
    accuracy = accuracy_score(y_test, y_pred)
    test_stats.append(accuracy)
    
# Calculate the mean
mean = np.mean(test_stats)
# Calculate the standard deviation
std = np.std(test_stats)

In [ ]:
# Confidence interval
n = len(test_stats)
confidence_level = 0.95
ci = t.interval(confidence_level, n - 1, loc=mean, scale=std)
print(f"Confidence interval: [{ci[0]:.3f}, {ci[1]:.3f}]")

# Print the results
print("Mean: {:.5f}".format(mean))
print("Standard deviation: {:.4f}".format(std))
plt.hist(test_stats, density=True)

In [ ]:
# Define the new value to test
sample_mean = np.mean(test_stats)
sample_std = np.std(test_stats, ddof=1)
sample_size = len(test_stats)

# Calculate the t-statistic and p-value
t_stat = (sample_mean - null_accuracy) / (sample_std)
p_value = 2 * stdtr(sample_size - 1, -np.abs(t_stat))

# Print the results
print("t-statistic:", t_stat)
print("p-value:", p_value)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
clf.fit(X_train, y_train)
# Make predictions on the test set
y_pred = clf.predict(X_test)
y_trainpred = clf.predict(X_train)
# Calculate the accuracy of the model
accuracyTrain = accuracy_score(y_train, y_trainpred)
accuracy = accuracy_score(y_test, y_pred)
print(accuracyTrain, "Train Accuracy")
print(accuracy, 'Test Accuracy')

label = list(df[category].unique())
label_t = []
for word in label:
    new_word = word + " True"
    label_t.append(new_word)
label_p = []
for word in label:
    new_word = word + " Pred"
    label_p.append(new_word)

# Compute the confusion matrix
conf_mat = confusion_matrix(y_test, y_pred, labels=label)
# Convert the matrix to a pandas DataFrame for easier manipulation
conf_df = pd.DataFrame(conf_mat, index=label_t, columns=label_p)
# Create a heatmap of the confusion matrix using seaborn
sb.heatmap(conf_df, annot=True, cmap='Blues', fmt='g')
# Generate the classification report
report = classification_report(y_test, y_pred)
print(report)

In [ ]:
# Define the number of folds
n_folds = 5

# Define the cross-validation method
kf = KFold(n_splits=n_folds, shuffle=True, random_state=42)
scores = cross_val_score(clf, X, y, cv=kf)
print(scores)
# Print the cross-validation scores
print(f"Cross-validation scores: {scores}")
print(f"Mean accuracy: {scores.mean():.3f}")
# Train the final model on the full dataset
clf.fit(X, y)

In [ ]:
from sklearn.model_selection import KFold, cross_val_predict
from sklearn.metrics import confusion_matrix
import pandas as pd

# Define the number of folds
n_folds = 5

# Define the cross-validation method
kf = KFold(n_splits=n_folds, shuffle=True, random_state=42)

# Get the predicted target values using cross_val_predict()
y_pred = cross_val_predict(clf, X, y, cv=kf)

# Get the confusion matrix using confusion_matrix()
cm = confusion_matrix(y, y_pred)

# Print the confusion matrix
print("Confusion matrix:")
print(cm)

# Compute the confusion matrix
conf_mat = confusion_matrix(y, y_pred, labels=label)
# Convert the matrix to a pandas DataFrame for easier manipulation
conf_df = pd.DataFrame(conf_mat, index=label_t, columns=label_p)
# Create a heatmap of the confusion matrix using seaborn
sb.heatmap(conf_df, annot=True, cmap='Blues', fmt='g')
# Generate the classification report
report = classification_report(y, y_pred)
print(report)

# 6 Families and **Hedge** (Family)(KNeighborsClassifier)(55.14%)

In [ ]:
category = "Family"
df = dfTrainHedge[dfTrainHedge['Family'].isin(['Sapindaceae', 'Rosaceae', 'Fagaceae','Cornaceae','Betulaceae','Pinaceae','HEDGE'])]
n_bootstrap = 1
# (467/1260)^2+(260/1260)^2+(231/1260)^2+(85/1260)^2+(78/1260)^2+(75/1260)^2+(64/1260)^2
null_accuracy = 0.2281
clf = KNeighborsClassifier(n_neighbors=5, weights='uniform', algorithm='auto', leaf_size=30, p=2, metric='minkowski', metric_params=None, n_jobs=None)
X = df.iloc[:, 14:].astype(np.float64)
y = df[category]
X

In [ ]:
counts = df.groupby(category).size().sort_values(ascending=False)
counts.plot(kind= "bar")
print("length:",len(dfTrain))
print(counts)

In [ ]:
# bootstrap
# Bootstrap the data and calculate the test statistic on each bootstrap sample
test_stats = []
for i in range(n_bootstrap):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
    clf.fit(X_train, y_train)
    # Make predictions on the test set
    y_pred = clf.predict(X_test)
    # Calculate the accuracy of the model
    accuracy = accuracy_score(y_test, y_pred)
    test_stats.append(accuracy)
    
# Calculate the mean
mean = np.mean(test_stats)
# Calculate the standard deviation
std = np.std(test_stats)

In [ ]:
# Confidence interval
n = len(test_stats)
confidence_level = 0.95
ci = t.interval(confidence_level, n - 1, loc=mean, scale=std)
print(f"Confidence interval: [{ci[0]:.3f}, {ci[1]:.3f}]")

# Print the results
print("Mean: {:.5f}".format(mean))
print("Standard deviation: {:.4f}".format(std))
plt.hist(test_stats, density=True)

In [ ]:
# Define the new value to test
sample_mean = np.mean(test_stats)
sample_std = np.std(test_stats, ddof=1)
sample_size = len(test_stats)

# Calculate the t-statistic and p-value
t_stat = (sample_mean - null_accuracy) / (sample_std)
p_value = 2 * stdtr(sample_size - 1, -np.abs(t_stat))

# Print the results
print("t-statistic:", t_stat)
print("p-value:", p_value)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
clf.fit(X_train, y_train)
# Make predictions on the test set
y_pred = clf.predict(X_test)
y_trainpred = clf.predict(X_train)
# Calculate the accuracy of the model
accuracyTrain = accuracy_score(y_train, y_trainpred)
accuracy = accuracy_score(y_test, y_pred)
print(accuracyTrain, "Train Accuracy")
print(accuracy, 'Test Accuracy')

label = list(df[category].unique())
label_t = []
for word in label:
    new_word = word + " True"
    label_t.append(new_word)
label_p = []
for word in label:
    new_word = word + " Pred"
    label_p.append(new_word)

# Compute the confusion matrix
conf_mat = confusion_matrix(y_test, y_pred, labels=label)
# Convert the matrix to a pandas DataFrame for easier manipulation
conf_df = pd.DataFrame(conf_mat, index=label_t, columns=label_p)
# Create a heatmap of the confusion matrix using seaborn
sb.heatmap(conf_df, annot=True, cmap='Blues', fmt='g')
# Generate the classification report
report = classification_report(y_test, y_pred)
print(report)

In [ ]:
# Define the number of folds
n_folds = 5

# Define the cross-validation method
kf = KFold(n_splits=n_folds, shuffle=True, random_state=42)
scores = cross_val_score(clf, X, y, cv=kf)
print(scores)
# Print the cross-validation scores
print(f"Cross-validation scores: {scores}")
print(f"Mean accuracy: {scores.mean():.3f}")
# Train the final model on the full dataset
clf.fit(X, y)

In [ ]:
from sklearn.model_selection import KFold, cross_val_predict
from sklearn.metrics import confusion_matrix
import pandas as pd

# Define the number of folds
n_folds = 5

# Define the cross-validation method
kf = KFold(n_splits=n_folds, shuffle=True, random_state=42)

# Get the predicted target values using cross_val_predict()
y_pred = cross_val_predict(clf, X, y, cv=kf)

# Get the confusion matrix using confusion_matrix()
cm = confusion_matrix(y, y_pred)

# Print the confusion matrix
print("Confusion matrix:")
print(cm)

# Compute the confusion matrix
conf_mat = confusion_matrix(y, y_pred, labels=label)
# Convert the matrix to a pandas DataFrame for easier manipulation
conf_df = pd.DataFrame(conf_mat, index=label_t, columns=label_p)
# Create a heatmap of the confusion matrix using seaborn
sb.heatmap(conf_df, annot=True, cmap='Blues', fmt='g')
# Generate the classification report
report = classification_report(y, y_pred)
print(report)

# 6 Families and **Hedge** (Family)(LinearDiscriminantAnalysis)(58%)(BAD)

In [ ]:
category = "Family"
df = dfTrainHedge[dfTrainHedge['Family'].isin(['Sapindaceae', 'Rosaceae', 'Fagaceae','Cornaceae','Betulaceae','Pinaceae','HEDGE'])]
n_bootstrap = 1
# (467/1260)^2+(260/1260)^2+(231/1260)^2+(85/1260)^2+(78/1260)^2+(75/1260)^2+(64/1260)^2
null_accuracy = 0.2281
clf = QuadraticDiscriminantAnalysis(priors=None, reg_param=0.0, store_covariance=False, tol=0.0001)
X = df.iloc[:, 14:].astype(np.float64)
y = df[category]
X

In [ ]:
counts = df.groupby(category).size().sort_values(ascending=False)
counts.plot(kind= "bar")
print("length:",len(dfTrain))
print(counts)

In [ ]:
# bootstrap
# Bootstrap the data and calculate the test statistic on each bootstrap sample
test_stats = []
for i in range(n_bootstrap):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
    clf.fit(X_train, y_train)
    # Make predictions on the test set
    y_pred = clf.predict(X_test)
    # Calculate the accuracy of the model
    accuracy = accuracy_score(y_test, y_pred)
    test_stats.append(accuracy)
    
# Calculate the mean
mean = np.mean(test_stats)
# Calculate the standard deviation
std = np.std(test_stats)

In [ ]:
# Confidence interval
n = len(test_stats)
confidence_level = 0.95
ci = t.interval(confidence_level, n - 1, loc=mean, scale=std)
print(f"Confidence interval: [{ci[0]:.3f}, {ci[1]:.3f}]")

# Print the results
print("Mean: {:.5f}".format(mean))
print("Standard deviation: {:.4f}".format(std))
plt.hist(test_stats, density=True)

In [ ]:
# Define the new value to test
sample_mean = np.mean(test_stats)
sample_std = np.std(test_stats, ddof=1)
sample_size = len(test_stats)

# Calculate the t-statistic and p-value
t_stat = (sample_mean - null_accuracy) / (sample_std)
p_value = 2 * stdtr(sample_size - 1, -np.abs(t_stat))

# Print the results
print("t-statistic:", t_stat)
print("p-value:", p_value)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
clf.fit(X_train, y_train)
# Make predictions on the test set
y_pred = clf.predict(X_test)
y_trainpred = clf.predict(X_train)
# Calculate the accuracy of the model
accuracyTrain = accuracy_score(y_train, y_trainpred)
accuracy = accuracy_score(y_test, y_pred)
print(accuracyTrain, "Train Accuracy")
print(accuracy, 'Test Accuracy')

label = list(df[category].unique())
label_t = []
for word in label:
    new_word = word + " True"
    label_t.append(new_word)
label_p = []
for word in label:
    new_word = word + " Pred"
    label_p.append(new_word)

# Compute the confusion matrix
conf_mat = confusion_matrix(y_test, y_pred, labels=label)
# Convert the matrix to a pandas DataFrame for easier manipulation
conf_df = pd.DataFrame(conf_mat, index=label_t, columns=label_p)
# Create a heatmap of the confusion matrix using seaborn
sb.heatmap(conf_df, annot=True, cmap='Blues', fmt='g')
# Generate the classification report
report = classification_report(y_test, y_pred)
print(report)

In [ ]:
# Define the number of folds
n_folds = 5

# Define the cross-validation method
kf = KFold(n_splits=n_folds, shuffle=True, random_state=42)
scores = cross_val_score(clf, X, y, cv=kf)
print(scores)
# Print the cross-validation scores
print(f"Cross-validation scores: {scores}")
print(f"Mean accuracy: {scores.mean():.3f}")
# Train the final model on the full dataset
clf.fit(X, y)

In [ ]:
from sklearn.model_selection import KFold, cross_val_predict
from sklearn.metrics import confusion_matrix
import pandas as pd

# Define the number of folds
n_folds = 5

# Define the cross-validation method
kf = KFold(n_splits=n_folds, shuffle=True, random_state=42)

# Get the predicted target values using cross_val_predict()
y_pred = cross_val_predict(clf, X, y, cv=kf)

# Get the confusion matrix using confusion_matrix()
cm = confusion_matrix(y, y_pred)

# Print the confusion matrix
print("Confusion matrix:")
print(cm)

# Compute the confusion matrix
conf_mat = confusion_matrix(y, y_pred, labels=label)
# Convert the matrix to a pandas DataFrame for easier manipulation
conf_df = pd.DataFrame(conf_mat, index=label_t, columns=label_p)
# Create a heatmap of the confusion matrix using seaborn
sb.heatmap(conf_df, annot=True, cmap='Blues', fmt='g')
# Generate the classification report
report = classification_report(y, y_pred)
print(report)

# 6 Families and **Hedge** (Family)(SVC)(38%) (BAD)

In [ ]:
category = "Family"
df = dfTrainHedge[dfTrainHedge['Family'].isin(['Sapindaceae', 'Rosaceae', 'Fagaceae','Cornaceae','Betulaceae','Pinaceae','HEDGE'])]
n_bootstrap = 1
# (467/1260)^2+(260/1260)^2+(231/1260)^2+(85/1260)^2+(78/1260)^2+(75/1260)^2+(64/1260)^2
null_accuracy = 0.2281
clf = SVC()
X = df.iloc[:, 14:].astype(np.float64)
y = df[category]
X

In [ ]:
counts = df.groupby(category).size().sort_values(ascending=False)
counts.plot(kind= "bar")
print("length:",len(dfTrain))
print(counts)

In [ ]:
# bootstrap
# Bootstrap the data and calculate the test statistic on each bootstrap sample
test_stats = []
for i in range(n_bootstrap):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
    clf.fit(X_train, y_train)
    # Make predictions on the test set
    y_pred = clf.predict(X_test)
    # Calculate the accuracy of the model
    accuracy = accuracy_score(y_test, y_pred)
    test_stats.append(accuracy)
    
# Calculate the mean
mean = np.mean(test_stats)
# Calculate the standard deviation
std = np.std(test_stats)

In [ ]:
# Confidence interval
n = len(test_stats)
confidence_level = 0.95
ci = t.interval(confidence_level, n - 1, loc=mean, scale=std)
print(f"Confidence interval: [{ci[0]:.3f}, {ci[1]:.3f}]")

# Print the results
print("Mean: {:.5f}".format(mean))
print("Standard deviation: {:.4f}".format(std))
plt.hist(test_stats, density=True)

In [ ]:
# Define the new value to test
sample_mean = np.mean(test_stats)
sample_std = np.std(test_stats, ddof=1)
sample_size = len(test_stats)

# Calculate the t-statistic and p-value
t_stat = (sample_mean - null_accuracy) / (sample_std)
p_value = 2 * stdtr(sample_size - 1, -np.abs(t_stat))

# Print the results
print("t-statistic:", t_stat)
print("p-value:", p_value)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
clf.fit(X_train, y_train)
# Make predictions on the test set
y_pred = clf.predict(X_test)
y_trainpred = clf.predict(X_train)
# Calculate the accuracy of the model
accuracyTrain = accuracy_score(y_train, y_trainpred)
accuracy = accuracy_score(y_test, y_pred)
print(accuracyTrain, "Train Accuracy")
print(accuracy, 'Test Accuracy')

label = list(df[category].unique())
label_t = []
for word in label:
    new_word = word + " True"
    label_t.append(new_word)
label_p = []
for word in label:
    new_word = word + " Pred"
    label_p.append(new_word)

# Compute the confusion matrix
conf_mat = confusion_matrix(y_test, y_pred, labels=label)
# Convert the matrix to a pandas DataFrame for easier manipulation
conf_df = pd.DataFrame(conf_mat, index=label_t, columns=label_p)
# Create a heatmap of the confusion matrix using seaborn
sb.heatmap(conf_df, annot=True, cmap='Blues', fmt='g')
# Generate the classification report
report = classification_report(y_test, y_pred)
print(report)

In [ ]:
# Define the number of folds
n_folds = 5

# Define the cross-validation method
kf = KFold(n_splits=n_folds, shuffle=True, random_state=42)
scores = cross_val_score(clf, X, y, cv=kf)
print(scores)
# Print the cross-validation scores
print(f"Cross-validation scores: {scores}")
print(f"Mean accuracy: {scores.mean():.3f}")
# Train the final model on the full dataset
clf.fit(X, y)

In [ ]:
from sklearn.model_selection import KFold, cross_val_predict
from sklearn.metrics import confusion_matrix
import pandas as pd

# Define the number of folds
n_folds = 5

# Define the cross-validation method
kf = KFold(n_splits=n_folds, shuffle=True, random_state=42)

# Get the predicted target values using cross_val_predict()
y_pred = cross_val_predict(clf, X, y, cv=kf)

# Get the confusion matrix using confusion_matrix()
cm = confusion_matrix(y, y_pred)

# Print the confusion matrix
print("Confusion matrix:")
print(cm)

# Compute the confusion matrix
conf_mat = confusion_matrix(y, y_pred, labels=label)
# Convert the matrix to a pandas DataFrame for easier manipulation
conf_df = pd.DataFrame(conf_mat, index=label_t, columns=label_p)
# Create a heatmap of the confusion matrix using seaborn
sb.heatmap(conf_df, annot=True, cmap='Blues', fmt='g')
# Generate the classification report
report = classification_report(y, y_pred)
print(report)

# 6 Families and **Hedge** (Family)(Oversample)
DecisionTreeClassifier

In [ ]:
from imblearn.over_sampling import SMOTE
category = "Family"
df = dfTrainHedge[dfTrainHedge['Family'].isin(['Sapindaceae', 'Rosaceae', 'Fagaceae','Cornaceae','Betulaceae','Pinaceae','HEDGE'])]
n_bootstrap = 1
# (467/1260)^2+(260/1260)^2+(231/1260)^2+(85/1260)^2+(78/1260)^2+(75/1260)^2+(64/1260)^2
null_accuracy = 0.2281
clf = DecisionTreeClassifier(criterion='gini', splitter='best', max_depth=8, min_samples_split=3)
# clf = DecisionTreeClassifier(criterion='entropy', splitter='random', max_depth=5, min_samples_split=5)
# clf = DecisionTreeClassifier(criterion='entropy', splitter='best', max_depth=8, min_samples_split=3)
X = df.iloc[:, 14:].astype(np.float64)
y = df[category]
X

In [ ]:
counts = df.groupby(category).size().sort_values(ascending=False)
counts.plot(kind= "bar")
print("length:",len(dfTrain))
print(counts)

In [ ]:
# bootstrap
# Bootstrap the data and calculate the test statistic on each bootstrap sample
test_stats = []
sm = SMOTE(random_state=42)
for i in range(n_bootstrap):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
    X_train_res, y_train_res = sm.fit_resample(X_train, y_train)
    clf.fit(X_train_res, y_train_res)
    # Make predictions on the test set
    y_pred = clf.predict(X_test)
    # Calculate the accuracy of the model
    accuracy = accuracy_score(y_test, y_pred)
    test_stats.append(accuracy)
    
# Calculate the mean
mean = np.mean(test_stats)
# Calculate the standard deviation
std = np.std(test_stats)

In [ ]:
# Confidence interval
n = len(test_stats)
confidence_level = 0.95
ci = t.interval(confidence_level, n - 1, loc=mean, scale=std)
print(f"Confidence interval: [{ci[0]:.3f}, {ci[1]:.3f}]")

# Print the results
print("Mean: {:.5f}".format(mean))
print("Standard deviation: {:.4f}".format(std))
plt.hist(test_stats, density=True)

In [ ]:
# Define the new value to test
sample_mean = np.mean(test_stats)
sample_std = np.std(test_stats, ddof=1)
sample_size = len(test_stats)

# Calculate the t-statistic and p-value
t_stat = (sample_mean - null_accuracy) / (sample_std)
p_value = 2 * stdtr(sample_size - 1, -np.abs(t_stat))

# Print the results
print("t-statistic:", t_stat)
print("p-value:", p_value)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
X_train_res, y_train_res = sm.fit_resample(X_train, y_train)
clf.fit(X_train_res, y_train_res)
# Make predictions on the test set
y_pred = clf.predict(X_test)
y_trainpred = clf.predict(X_train)
# Calculate the accuracy of the model
accuracyTrain = accuracy_score(y_train, y_trainpred)
accuracy = accuracy_score(y_test, y_pred)
print(accuracyTrain, "Train Accuracy")
print(accuracy, 'Test Accuracy')

label = list(df[category].unique())
label_t = []
for word in label:
    new_word = word + " True"
    label_t.append(new_word)
label_p = []
for word in label:
    new_word = word + " Pred"
    label_p.append(new_word)

# Compute the confusion matrix
conf_mat = confusion_matrix(y_test, y_pred, labels=label)
# Convert the matrix to a pandas DataFrame for easier manipulation
conf_df = pd.DataFrame(conf_mat, index=label_t, columns=label_p)
# Create a heatmap of the confusion matrix using seaborn
sb.heatmap(conf_df, annot=True, cmap='Blues', fmt='g')
# Generate the classification report
report = classification_report(y_test, y_pred)
print(report)
importances = clf.feature_importances_
print(importances)

In [ ]:
# Define the number of folds
n_folds = 5

# Define the cross-validation method
kf = KFold(n_splits=n_folds, shuffle=True, random_state=42)
scores = cross_val_score(clf, X, y, cv=kf)
print(scores)
# Print the cross-validation scores
print(f"Cross-validation scores: {scores}")
print(f"Mean accuracy: {scores.mean():.3f}")
# Train the final model on the full dataset
clf.fit(X, y)

In [ ]:
from sklearn.model_selection import KFold, cross_val_predict
from sklearn.metrics import confusion_matrix
import pandas as pd

# Define the number of folds
n_folds = 5

# Define the cross-validation method
kf = KFold(n_splits=n_folds, shuffle=True, random_state=42)

# Get the predicted target values using cross_val_predict()
y_pred = cross_val_predict(clf, X, y, cv=kf)

# Get the confusion matrix using confusion_matrix()
cm = confusion_matrix(y, y_pred)

# Print the confusion matrix
print("Confusion matrix:")
print(cm)

# Compute the confusion matrix
conf_mat = confusion_matrix(y, y_pred, labels=label)
# Convert the matrix to a pandas DataFrame for easier manipulation
conf_df = pd.DataFrame(conf_mat, index=label_t, columns=label_p)
# Create a heatmap of the confusion matrix using seaborn
sb.heatmap(conf_df, annot=True, cmap='Blues', fmt='g')
# Generate the classification report
report = classification_report(y, y_pred)
print(report)

# 6 Families and **Hedge** (Family)(Oversample)(SVC)
(Insignificant,BAD)

In [ ]:
category = "Family"
df = dfTrainHedge[dfTrainHedge['Family'].isin(['Sapindaceae', 'Rosaceae', 'Fagaceae','Cornaceae','Betulaceae','Pinaceae','HEDGE'])]
n_bootstrap = 1
# (467/1260)^2+(260/1260)^2+(231/1260)^2+(85/1260)^2+(78/1260)^2+(75/1260)^2+(64/1260)^2
null_accuracy = 0.2281
clf = SVC(C=1.0, kernel='rbf', degree=3, gamma='scale', coef0=0.0, shrinking=True, probability=False, tol=0.001, cache_size=200, class_weight=None, verbose=False, max_iter=-1, decision_function_shape='ovr', break_ties=False, random_state=None)
# clf = DecisionTreeClassifier(criterion='entropy', splitter='random', max_depth=5, min_samples_split=5)
# clf = DecisionTreeClassifier(criterion='entropy', splitter='best', max_depth=8, min_samples_split=3)
X = df.iloc[:, 14:].astype(np.float64)
y = df[category]
X

In [ ]:
counts = df.groupby(category).size().sort_values(ascending=False)
counts.plot(kind= "bar")
print("length:",len(dfTrain))
print(counts)

In [ ]:
# bootstrap
# Bootstrap the data and calculate the test statistic on each bootstrap sample
test_stats = []
sm = SMOTE(random_state=42)
for i in range(n_bootstrap):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
    X_train_res, y_train_res = sm.fit_resample(X_train, y_train)
    clf.fit(X_train_res, y_train_res)
    # Make predictions on the test set
    y_pred = clf.predict(X_test)
    # Calculate the accuracy of the model
    accuracy = accuracy_score(y_test, y_pred)
    test_stats.append(accuracy)
    
# Calculate the mean
mean = np.mean(test_stats)
# Calculate the standard deviation
std = np.std(test_stats)

In [ ]:
# Confidence interval
n = len(test_stats)
confidence_level = 0.95
ci = t.interval(confidence_level, n - 1, loc=mean, scale=std)
print(f"Confidence interval: [{ci[0]:.3f}, {ci[1]:.3f}]")

# Print the results
print("Mean: {:.5f}".format(mean))
print("Standard deviation: {:.4f}".format(std))
plt.hist(test_stats, density=True)

In [ ]:
# Define the new value to test
sample_mean = np.mean(test_stats)
sample_std = np.std(test_stats, ddof=1)
sample_size = len(test_stats)

# Calculate the t-statistic and p-value
t_stat = (sample_mean - null_accuracy) / (sample_std)
p_value = 2 * stdtr(sample_size - 1, -np.abs(t_stat))

# Print the results
print("t-statistic:", t_stat)
print("p-value:", p_value)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
X_train_res, y_train_res = sm.fit_resample(X_train, y_train)
clf.fit(X_train_res, y_train_res)
# Make predictions on the test set
y_pred = clf.predict(X_test)
y_trainpred = clf.predict(X_train)
# Calculate the accuracy of the model
accuracyTrain = accuracy_score(y_train, y_trainpred)
accuracy = accuracy_score(y_test, y_pred)
print(accuracyTrain, "Train Accuracy")
print(accuracy, 'Test Accuracy')

label = list(df[category].unique())
label_t = []
for word in label:
    new_word = word + " True"
    label_t.append(new_word)
label_p = []
for word in label:
    new_word = word + " Pred"
    label_p.append(new_word)

# Compute the confusion matrix
conf_mat = confusion_matrix(y_test, y_pred, labels=label)
# Convert the matrix to a pandas DataFrame for easier manipulation
conf_df = pd.DataFrame(conf_mat, index=label_t, columns=label_p)
# Create a heatmap of the confusion matrix using seaborn
sb.heatmap(conf_df, annot=True, cmap='Blues', fmt='g')
# Generate the classification report
report = classification_report(y_test, y_pred)
print(report)

In [ ]:
# Define the number of folds
n_folds = 5

# Define the cross-validation method
kf = KFold(n_splits=n_folds, shuffle=True, random_state=42)
scores = cross_val_score(clf, X, y, cv=kf)
print(scores)
# Print the cross-validation scores
print(f"Cross-validation scores: {scores}")
print(f"Mean accuracy: {scores.mean():.3f}")
# Train the final model on the full dataset
clf.fit(X, y)

In [ ]:
from sklearn.model_selection import KFold, cross_val_predict
from sklearn.metrics import confusion_matrix
import pandas as pd

# Define the number of folds
n_folds = 5

# Define the cross-validation method
kf = KFold(n_splits=n_folds, shuffle=True, random_state=42)

# Get the predicted target values using cross_val_predict()
y_pred = cross_val_predict(clf, X, y, cv=kf)

# Get the confusion matrix using confusion_matrix()
cm = confusion_matrix(y, y_pred)

# Print the confusion matrix
print("Confusion matrix:")
print(cm)

# Compute the confusion matrix
conf_mat = confusion_matrix(y, y_pred, labels=label)
# Convert the matrix to a pandas DataFrame for easier manipulation
conf_df = pd.DataFrame(conf_mat, index=label_t, columns=label_p)
# Create a heatmap of the confusion matrix using seaborn
sb.heatmap(conf_df, annot=True, cmap='Blues', fmt='g')
# Generate the classification report
report = classification_report(y, y_pred)
print(report)

# 6 Families and **Hedge** (Family)(Oversample)(RandomForestClassifier)
RandomForestClassifier
200 estimators

In [ ]:
category = "Family"
df = dfTrainHedge[dfTrainHedge['Family'].isin(['Sapindaceae', 'Rosaceae', 'Fagaceae','Cornaceae','Betulaceae','Pinaceae','HEDGE'])]
n_bootstrap = 2
# (467/1260)^2+(260/1260)^2+(231/1260)^2+(85/1260)^2+(78/1260)^2+(75/1260)^2+(64/1260)^2
null_accuracy = 0.2281
clf = RandomForestClassifier(n_estimators=100, criterion='gini')
X = df.iloc[:, 14:].astype(np.float64)
y = df[category]
X

In [ ]:
counts = df.groupby(category).size().sort_values(ascending=False)
counts.plot(kind= "bar")
print("length:",len(dfTrain))
print(counts)

In [ ]:
# bootstrap
# Bootstrap the data and calculate the test statistic on each bootstrap sample
test_stats = []
sm = SMOTE(random_state=42)
for i in range(n_bootstrap):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
    X_train_res, y_train_res = sm.fit_resample(X_train, y_train)
    clf.fit(X_train_res, y_train_res)
    # Make predictions on the test set
    y_pred = clf.predict(X_test)
    # Calculate the accuracy of the model
    accuracy = accuracy_score(y_test, y_pred)
    test_stats.append(accuracy)
    
# Calculate the mean
mean = np.mean(test_stats)
# Calculate the standard deviation
std = np.std(test_stats)

In [ ]:
# Confidence interval
n = len(test_stats)
confidence_level = 0.95
ci = t.interval(confidence_level, n - 1, loc=mean, scale=std)
print(f"Confidence interval: [{ci[0]:.3f}, {ci[1]:.3f}]")

# Print the results
print("Mean: {:.5f}".format(mean))
print("Standard deviation: {:.4f}".format(std))
plt.hist(test_stats, density=True)

In [ ]:
# Define the new value to test
sample_mean = np.mean(test_stats)
sample_std = np.std(test_stats, ddof=1)
sample_size = len(test_stats)

# Calculate the t-statistic and p-value
t_stat = (sample_mean - null_accuracy) / (sample_std)
p_value = 2 * stdtr(sample_size - 1, -np.abs(t_stat))

# Print the results
print("t-statistic:", t_stat)
print("p-value:", p_value)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
X_train_res, y_train_res = sm.fit_resample(X_train, y_train)
clf.fit(X_train_res, y_train_res)
# Make predictions on the test set
y_pred = clf.predict(X_test)
y_trainpred = clf.predict(X_train)
# Calculate the accuracy of the model
accuracyTrain = accuracy_score(y_train, y_trainpred)
accuracy = accuracy_score(y_test, y_pred)
print(accuracyTrain, "Train Accuracy")
print(accuracy, 'Test Accuracy')

label = list(df[category].unique())
label_t = []
for word in label:
    new_word = word + " True"
    label_t.append(new_word)
label_p = []
for word in label:
    new_word = word + " Pred"
    label_p.append(new_word)

# Compute the confusion matrix
conf_mat = confusion_matrix(y_test, y_pred, labels=label)
# Convert the matrix to a pandas DataFrame for easier manipulation
conf_df = pd.DataFrame(conf_mat, index=label_t, columns=label_p)
# Create a heatmap of the confusion matrix using seaborn
sb.heatmap(conf_df, annot=True, cmap='Blues', fmt='g')
# Generate the classification report
report = classification_report(y_test, y_pred)
print(report)

In [ ]:
# Define the number of folds
n_folds = 5

# Define the cross-validation method
kf = KFold(n_splits=n_folds, shuffle=True, random_state=42)
scores = cross_val_score(clf, X, y, cv=kf)
print(scores)
# Print the cross-validation scores
print(f"Cross-validation scores: {scores}")
print(f"Mean accuracy: {scores.mean():.3f}")
# Train the final model on the full dataset
clf.fit(X, y)

In [ ]:
# Define the number of folds
n_folds = 5

# Define the cross-validation method
kf = KFold(n_splits=n_folds, shuffle=True, random_state=42)

# Get the predicted target values using cross_val_predict()
y_pred = cross_val_predict(clf, X, y, cv=kf)

# Get the confusion matrix using confusion_matrix()
cm = confusion_matrix(y, y_pred)

# Print the confusion matrix
print("Confusion matrix:")
print(cm)

# Compute the confusion matrix
conf_mat = confusion_matrix(y, y_pred, labels=label)
# Convert the matrix to a pandas DataFrame for easier manipulation
conf_df = pd.DataFrame(conf_mat, index=label_t, columns=label_p)
# Create a heatmap of the confusion matrix using seaborn
sb.heatmap(conf_df, annot=True, cmap='Blues', fmt='g')
# Generate the classification report
report = classification_report(y, y_pred)
print(report)

In [ ]:
# Get the feature importance scores
importance_scores = clf.feature_importances_
feature_names = df.iloc[:, 15:].columns[:-1]
# Print the feature importance scores
for feature_name, importance_score in zip(feature_names, importance_scores):
    print(f'{feature_name}: {importance_score:.4f}')

## Apply the results to predict population

# Functional and **Hedge** (Functional)(Oversample)(RandomForestClassifier)


In [ ]:
category = "Functional"
df = dfTrainHedge
n_bootstrap = 2
null_accuracy = 0.6785
clf = RandomForestClassifier(n_estimators=100, criterion='gini')
X = df.iloc[:, 14:].astype(np.float64)
y = df[category]
X

In [ ]:
counts = df.groupby(category).size().sort_values(ascending=False)
counts.plot(kind= "bar")
print("length:",len(df))
print(counts)

In [ ]:
# bootstrap
# Bootstrap the data and calculate the test statistic on each bootstrap sample
test_stats = []
sm = SMOTE(random_state=42)
for i in range(n_bootstrap):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
    X_train_res, y_train_res = sm.fit_resample(X_train, y_train)
    clf.fit(X_train_res, y_train_res)
    # Make predictions on the test set
    y_pred = clf.predict(X_test)
    # Calculate the accuracy of the model
    accuracy = accuracy_score(y_test, y_pred)
    test_stats.append(accuracy)
    
# Calculate the mean
mean = np.mean(test_stats)
# Calculate the standard deviation
std = np.std(test_stats)

In [ ]:
# Confidence interval
n = len(test_stats)
confidence_level = 0.95
ci = t.interval(confidence_level, n - 1, loc=mean, scale=std)
print(f"Confidence interval: [{ci[0]:.3f}, {ci[1]:.3f}]")

# Print the results
print("Mean: {:.5f}".format(mean))
print("Standard deviation: {:.4f}".format(std))
plt.hist(test_stats, density=True)

In [ ]:
# Define the new value to test
sample_mean = np.mean(test_stats)
sample_std = np.std(test_stats, ddof=1)
sample_size = len(test_stats)

# Calculate the t-statistic and p-value
t_stat = (sample_mean - null_accuracy) / (sample_std)
p_value = 2 * stdtr(sample_size - 1, -np.abs(t_stat))

# Print the results
print("t-statistic:", t_stat)
print("p-value:", p_value)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
X_train_res, y_train_res = sm.fit_resample(X_train, y_train)
clf.fit(X_train_res, y_train_res)
# Make predictions on the test set
y_pred = clf.predict(X_test)
y_trainpred = clf.predict(X_train)
# Calculate the accuracy of the model
accuracyTrain = accuracy_score(y_train, y_trainpred)
accuracy = accuracy_score(y_test, y_pred)
print(accuracyTrain, "Train Accuracy")
print(accuracy, 'Test Accuracy')

label = list(df[category].unique())
label_t = []
for word in label:
    new_word = word + " True"
    label_t.append(new_word)
label_p = []
for word in label:
    new_word = word + " Pred"
    label_p.append(new_word)

# Compute the confusion matrix
conf_mat = confusion_matrix(y_test, y_pred, labels=label)
# Convert the matrix to a pandas DataFrame for easier manipulation
conf_df = pd.DataFrame(conf_mat, index=label_t, columns=label_p)
# Create a heatmap of the confusion matrix using seaborn
sb.heatmap(conf_df, annot=True, cmap='Blues', fmt='g')
# Generate the classification report
report = classification_report(y_test, y_pred)
print(report)

In [ ]:
# Define the number of folds
n_folds = 5

# Define the cross-validation method
kf = KFold(n_splits=n_folds, shuffle=True, random_state=42)
scores = cross_val_score(clf, X, y, cv=kf)
print(scores)
# Print the cross-validation scores
print(f"Cross-validation scores: {scores}")
print(f"Mean accuracy: {scores.mean():.3f}")
# Train the final model on the full dataset
clf.fit(X, y)

In [ ]:
# Define the number of folds
n_folds = 5

# Define the cross-validation method
kf = KFold(n_splits=n_folds, shuffle=True, random_state=42)

# Get the predicted target values using cross_val_predict()
y_pred = cross_val_predict(clf, X, y, cv=kf)

# Get the confusion matrix using confusion_matrix()
cm = confusion_matrix(y, y_pred)

# Print the confusion matrix
print("Confusion matrix:")
print(cm)

# Compute the confusion matrix
conf_mat = confusion_matrix(y, y_pred, labels=label)
# Convert the matrix to a pandas DataFrame for easier manipulation
conf_df = pd.DataFrame(conf_mat, index=label_t, columns=label_p)
# Create a heatmap of the confusion matrix using seaborn
sb.heatmap(conf_df, annot=True, cmap='Blues', fmt='g')
# Generate the classification report
report = classification_report(y, y_pred)
print(report)

In [ ]:
# Get the feature importance scores
importance_scores = clf.feature_importances_
feature_names = df.iloc[:, 15:].columns[:-1]
# Print the feature importance scores
for feature_name, importance_score in zip(feature_names, importance_scores):
    print(f'{feature_name}: {importance_score:.4f}')

## Apply the results to predict population

# 6 Families and **Hedge** (Family)(Oversample)(LDA)


In [ ]:
category = "Family"
df = dfTrainHedge[dfTrainHedge['Family'].isin(['Sapindaceae', 'Rosaceae', 'Fagaceae','Cornaceae','Betulaceae','Pinaceae','HEDGE'])]
n_bootstrap = 1
# (467/1260)^2+(260/1260)^2+(231/1260)^2+(85/1260)^2+(78/1260)^2+(75/1260)^2+(64/1260)^2
null_accuracy = 0.2281
clf = LinearDiscriminantAnalysis(solver='lsqr', shrinkage=None, priors=None, n_components=None, store_covariance=False, tol=0.0001, covariance_estimator=None)
X = df.iloc[:, 14:].astype(np.float64)
y = df[category]
X

In [ ]:
counts = df.groupby(category).size().sort_values(ascending=False)
counts.plot(kind= "bar")
print("length:",len(dfTrain))
print(counts)

In [ ]:
# bootstrap
# Bootstrap the data and calculate the test statistic on each bootstrap sample
test_stats = []
sm = SMOTE(random_state=42)
for i in range(n_bootstrap):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
    X_train_res, y_train_res = sm.fit_resample(X_train, y_train)
    clf.fit(X_train_res, y_train_res)
    # Make predictions on the test set
    y_pred = clf.predict(X_test)
    # Calculate the accuracy of the model
    accuracy = accuracy_score(y_test, y_pred)
    test_stats.append(accuracy)
    
# Calculate the mean
mean = np.mean(test_stats)
# Calculate the standard deviation
std = np.std(test_stats)

In [ ]:
# Confidence interval
n = len(test_stats)
confidence_level = 0.95
ci = t.interval(confidence_level, n - 1, loc=mean, scale=std)
print(f"Confidence interval: [{ci[0]:.3f}, {ci[1]:.3f}]")

# Print the results
print("Mean: {:.5f}".format(mean))
print("Standard deviation: {:.4f}".format(std))
plt.hist(test_stats, density=True)

In [ ]:
# Define the new value to test
sample_mean = np.mean(test_stats)
sample_std = np.std(test_stats, ddof=1)
sample_size = len(test_stats)

# Calculate the t-statistic and p-value
t_stat = (sample_mean - null_accuracy) / (sample_std)
p_value = 2 * stdtr(sample_size - 1, -np.abs(t_stat))

# Print the results
print("t-statistic:", t_stat)
print("p-value:", p_value)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
X_train_res, y_train_res = sm.fit_resample(X_train, y_train)
clf.fit(X_train_res, y_train_res)
# Make predictions on the test set
y_pred = clf.predict(X_test)
y_trainpred = clf.predict(X_train)
# Calculate the accuracy of the model
accuracyTrain = accuracy_score(y_train, y_trainpred)
accuracy = accuracy_score(y_test, y_pred)
print(accuracyTrain, "Train Accuracy")
print(accuracy, 'Test Accuracy')

label = list(df[category].unique())
label_t = []
for word in label:
    new_word = word + " True"
    label_t.append(new_word)
label_p = []
for word in label:
    new_word = word + " Pred"
    label_p.append(new_word)

# Compute the confusion matrix
conf_mat = confusion_matrix(y_test, y_pred, labels=label)
# Convert the matrix to a pandas DataFrame for easier manipulation
conf_df = pd.DataFrame(conf_mat, index=label_t, columns=label_p)
# Create a heatmap of the confusion matrix using seaborn
sb.heatmap(conf_df, annot=True, cmap='Blues', fmt='g')
# Generate the classification report
report = classification_report(y_test, y_pred)
print(report)

In [ ]:
# Define the number of folds
n_folds = 5

# Define the cross-validation method
kf = KFold(n_splits=n_folds, shuffle=True, random_state=42)
scores = cross_val_score(clf, X, y, cv=kf)
print(scores)
# Print the cross-validation scores
print(f"Cross-validation scores: {scores}")
print(f"Mean accuracy: {scores.mean():.3f}")
# Train the final model on the full dataset
clf.fit(X, y)

In [ ]:
# Define the number of folds
n_folds = 5

# Define the cross-validation method
kf = KFold(n_splits=n_folds, shuffle=True, random_state=42)

# Get the predicted target values using cross_val_predict()
y_pred = cross_val_predict(clf, X, y, cv=kf)

# Get the confusion matrix using confusion_matrix()
cm = confusion_matrix(y, y_pred)

# Print the confusion matrix
print("Confusion matrix:")
print(cm)

# Compute the confusion matrix
conf_mat = confusion_matrix(y, y_pred, labels=label)
# Convert the matrix to a pandas DataFrame for easier manipulation
conf_df = pd.DataFrame(conf_mat, index=label_t, columns=label_p)
# Create a heatmap of the confusion matrix using seaborn
sb.heatmap(conf_df, annot=True, cmap='Blues', fmt='g')
# Generate the classification report
report = classification_report(y, y_pred)
print(report)

In [ ]:
# # Get the feature importance scores
# importance_scores = clf.feature_importances_
# feature_names = df.iloc[:, 15:].columns[:-1]
# # Print the feature importance scores
# for feature_name, importance_score in zip(feature_names, importance_scores):
#     print(f'{feature_name}: {importance_score:.4f}')

# **Selected Model**

# Coniferous/Deciduous and **Hedge** (Functional)(Oversample)(RandomForestClassifier)
Null: 0.6785
Result: 


In [ ]:
category = "Functional"
df = dfTrainHedge
n_bootstrap = 1
null_accuracy = 0.6785
clf = RandomForestClassifier(n_estimators=100, criterion='gini')
X = df.iloc[:, 14:].astype(np.float64)
y = df[category]
X

In [ ]:
counts = df.groupby(category).size().sort_values(ascending=False)
counts.plot(kind= "bar")
print("length:",len(df))
print(counts)

In [ ]:
# bootstrap
# Bootstrap the data and calculate the test statistic on each bootstrap sample
test_stats = []
sm = SMOTE(random_state=42)
for i in range(n_bootstrap):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
    X_train_res, y_train_res = sm.fit_resample(X_train, y_train)
    clf.fit(X_train_res, y_train_res)
    # Make predictions on the test set
    y_pred = clf.predict(X_test)
    # Calculate the accuracy of the model
    accuracy = accuracy_score(y_test, y_pred)
    test_stats.append(accuracy)
    
# Calculate the mean
mean = np.mean(test_stats)
# Calculate the standard deviation
std = np.std(test_stats)

In [ ]:
# Confidence interval
n = len(test_stats)
confidence_level = 0.95
ci = t.interval(confidence_level, n - 1, loc=mean, scale=std)
print(f"Confidence interval: [{ci[0]:.3f}, {ci[1]:.3f}]")

# Print the results
print("Mean: {:.5f}".format(mean))
print("Standard deviation: {:.4f}".format(std))
plt.hist(test_stats, density=True)

In [ ]:
# Define the new value to test
sample_mean = np.mean(test_stats)
sample_std = np.std(test_stats, ddof=1)
sample_size = len(test_stats)

# Calculate the t-statistic and p-value
t_stat = (sample_mean - null_accuracy) / (sample_std)
p_value = 2 * stdtr(sample_size - 1, -np.abs(t_stat))

# Print the results
print("t-statistic:", t_stat)
print("p-value:", p_value)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
X_train_res, y_train_res = sm.fit_resample(X_train, y_train)
clf.fit(X_train_res, y_train_res)
# Make predictions on the test set
y_pred = clf.predict(X_test)
y_trainpred = clf.predict(X_train)
# Calculate the accuracy of the model
accuracyTrain = accuracy_score(y_train, y_trainpred)
accuracy = accuracy_score(y_test, y_pred)
print(accuracyTrain, "Train Accuracy")
print(accuracy, 'Test Accuracy')

label = list(df[category].unique())
label_t = []
for word in label:
    new_word = word + " True"
    label_t.append(new_word)
label_p = []
for word in label:
    new_word = word + " Pred"
    label_p.append(new_word)

# Compute the confusion matrix
conf_mat = confusion_matrix(y_test, y_pred, labels=label)
# Convert the matrix to a pandas DataFrame for easier manipulation
conf_df = pd.DataFrame(conf_mat, index=label_t, columns=label_p)
# Create a heatmap of the confusion matrix using seaborn
sb.heatmap(conf_df, annot=True, cmap='Blues', fmt='g')
# Generate the classification report
report = classification_report(y_test, y_pred)
print(report)

In [ ]:
# Define the number of folds
n_folds = 5

# Define the cross-validation method
kf = KFold(n_splits=n_folds, shuffle=True, random_state=42)
scores = cross_val_score(clf, X, y, cv=kf)
print(scores)
# Print the cross-validation scores
print(f"Cross-validation scores: {scores}")
print(f"Mean accuracy: {scores.mean():.3f}")
# Train the final model on the full dataset
clf.fit(X, y)

In [ ]:
# Define the number of folds
n_folds = 5

# Define the cross-validation method
kf = KFold(n_splits=n_folds, shuffle=True, random_state=42)

# Get the predicted target values using cross_val_predict()
y_pred = cross_val_predict(clf, X, y, cv=kf)

# Get the confusion matrix using confusion_matrix()
cm = confusion_matrix(y, y_pred)

# Print the confusion matrix
print("Confusion matrix:")
print(cm)

# Compute the confusion matrix
conf_mat = confusion_matrix(y, y_pred, labels=label)
# Convert the matrix to a pandas DataFrame for easier manipulation
conf_df = pd.DataFrame(conf_mat, index=label_t, columns=label_p)
# Create a heatmap of the confusion matrix using seaborn
sb.heatmap(conf_df, annot=True, cmap='Blues', fmt='g')
# Generate the classification report
report = classification_report(y, y_pred)
print(report)

In [ ]:
# Get the feature importance scores
importance_scores = clf.feature_importances_
feature_names = df.iloc[:, 15:].columns[:-1]
# Print the feature importance scores
for feature_name, importance_score in zip(feature_names, importance_scores):
    print(f'{feature_name}: {importance_score:.4f}')

In [ ]:
# Use the model to make predictions on the data
predictions = clf.predict(population_independent)

# Add the predictions as a new column to the DataFrame
popu = popu.assign(Functional=predictions)
popu

## Apply the results to predict population

# Good/Fair/Bad and **Hedge** (Condition)(Oversample)(RandomForestClassifier)
Null: 0.4338
Result: 

In [ ]:
category = "Condition"
df = dfTrain
n_bootstrap = 1
null_accuracy = 0.4338
clf = RandomForestClassifier(n_estimators=100, criterion='gini')
X = df.iloc[:, 14:].astype(np.float64)
y = df[category]
X

In [ ]:
counts = df.groupby(category).size().sort_values(ascending=False)
counts.plot(kind= "bar")
print("length:",len(df))
print(counts)

In [ ]:
# bootstrap
# Bootstrap the data and calculate the test statistic on each bootstrap sample
test_stats = []
sm = SMOTE(random_state=42)
for i in range(n_bootstrap):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
    X_train_res, y_train_res = sm.fit_resample(X_train, y_train)
    clf.fit(X_train_res, y_train_res)
    # Make predictions on the test set
    y_pred = clf.predict(X_test)
    # Calculate the accuracy of the model
    accuracy = accuracy_score(y_test, y_pred)
    test_stats.append(accuracy)
    
# Calculate the mean
mean = np.mean(test_stats)
# Calculate the standard deviation
std = np.std(test_stats)

In [ ]:
# Confidence interval
n = len(test_stats)
confidence_level = 0.95
ci = t.interval(confidence_level, n - 1, loc=mean, scale=std)
print(f"Confidence interval: [{ci[0]:.3f}, {ci[1]:.3f}]")

# Print the results
print("Mean: {:.5f}".format(mean))
print("Standard deviation: {:.4f}".format(std))
plt.hist(test_stats, density=True)

In [ ]:
# Define the new value to test
sample_mean = np.mean(test_stats)
sample_std = np.std(test_stats, ddof=1)
sample_size = len(test_stats)

# Calculate the t-statistic and p-value
t_stat = (sample_mean - null_accuracy) / (sample_std)
p_value = 2 * stdtr(sample_size - 1, -np.abs(t_stat))

# Print the results
print("t-statistic:", t_stat)
print("p-value:", p_value)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
X_train_res, y_train_res = sm.fit_resample(X_train, y_train)
clf.fit(X_train_res, y_train_res)
# Make predictions on the test set
y_pred = clf.predict(X_test)
y_trainpred = clf.predict(X_train)
# Calculate the accuracy of the model
accuracyTrain = accuracy_score(y_train, y_trainpred)
accuracy = accuracy_score(y_test, y_pred)
print(accuracyTrain, "Train Accuracy")
print(accuracy, 'Test Accuracy')

label = list(df[category].unique())
label_t = []
for word in label:
    new_word = word + " True"
    label_t.append(new_word)
label_p = []
for word in label:
    new_word = word + " Pred"
    label_p.append(new_word)

# Compute the confusion matrix
conf_mat = confusion_matrix(y_test, y_pred, labels=label)
# Convert the matrix to a pandas DataFrame for easier manipulation
conf_df = pd.DataFrame(conf_mat, index=label_t, columns=label_p)
# Create a heatmap of the confusion matrix using seaborn
sb.heatmap(conf_df, annot=True, cmap='Blues', fmt='g')
# Generate the classification report
report = classification_report(y_test, y_pred)
print(report)

In [ ]:
# Define the number of folds
n_folds = 5

# Define the cross-validation method
kf = KFold(n_splits=n_folds, shuffle=True, random_state=42)
scores = cross_val_score(clf, X, y, cv=kf)
print(scores)
# Print the cross-validation scores
print(f"Cross-validation scores: {scores}")
print(f"Mean accuracy: {scores.mean():.3f}")
# Train the final model on the full dataset
clf.fit(X, y)

In [ ]:
# Define the number of folds
n_folds = 5

# Define the cross-validation method
kf = KFold(n_splits=n_folds, shuffle=True, random_state=42)

# Get the predicted target values using cross_val_predict()
y_pred = cross_val_predict(clf, X, y, cv=kf)

# Get the confusion matrix using confusion_matrix()
cm = confusion_matrix(y, y_pred)

# Print the confusion matrix
print("Confusion matrix:")
print(cm)

# Compute the confusion matrix
conf_mat = confusion_matrix(y, y_pred, labels=label)
# Convert the matrix to a pandas DataFrame for easier manipulation
conf_df = pd.DataFrame(conf_mat, index=label_t, columns=label_p)
# Create a heatmap of the confusion matrix using seaborn
sb.heatmap(conf_df, annot=True, cmap='Blues', fmt='g')
# Generate the classification report
report = classification_report(y, y_pred)
print(report)

In [ ]:
# Get the feature importance scores
importance_scores = clf.feature_importances_
feature_names = df.iloc[:, 15:].columns[:-1]
# Print the feature importance scores
for feature_name, importance_score in zip(feature_names, importance_scores):
    print(f'{feature_name}: {importance_score:.4f}')

In [ ]:
# Use the model to make predictions on the data
predictions = clf.predict(population_independent)

# Add the predictions as a new column to the DataFrame
popu = popu.assign(Condition=predictions)
popu

## Apply the results to predict population

# 6 Families and **Hedge** (Family)(Oversample)(RandomForestClassifier)
Null: 0.2281
Results: 
200 estimators

In [ ]:
category = "Family"
df = dfTrainHedge[dfTrainHedge['Family'].isin(['Sapindaceae', 'Rosaceae', 'Fagaceae','Cornaceae','Betulaceae','Pinaceae','HEDGE'])]
n_bootstrap = 1
# (467/1260)^2+(260/1260)^2+(231/1260)^2+(85/1260)^2+(78/1260)^2+(75/1260)^2+(64/1260)^2
null_accuracy = 0.2281
clf = RandomForestClassifier(n_estimators=100, criterion='gini')
X = df.iloc[:, 14:].astype(np.float64)
y = df[category]
X

In [ ]:
counts = df.groupby(category).size().sort_values(ascending=False)
counts.plot(kind= "bar")
print("length:",len(dfTrain))
print(counts)

In [ ]:
# bootstrap
# Bootstrap the data and calculate the test statistic on each bootstrap sample
test_stats = []
sm = SMOTE(random_state=42)
for i in range(n_bootstrap):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
    X_train_res, y_train_res = sm.fit_resample(X_train, y_train)
    clf.fit(X_train_res, y_train_res)
    # Make predictions on the test set
    y_pred = clf.predict(X_test)
    # Calculate the accuracy of the model
    accuracy = accuracy_score(y_test, y_pred)
    test_stats.append(accuracy)
    
# Calculate the mean
mean = np.mean(test_stats)
# Calculate the standard deviation
std = np.std(test_stats)

In [ ]:
# Confidence interval
n = len(test_stats)
confidence_level = 0.95
ci = t.interval(confidence_level, n - 1, loc=mean, scale=std)
print(f"Confidence interval: [{ci[0]:.3f}, {ci[1]:.3f}]")

# Print the results
print("Mean: {:.5f}".format(mean))
print("Standard deviation: {:.4f}".format(std))
plt.hist(test_stats, density=True)

In [ ]:
# Define the new value to test
sample_mean = np.mean(test_stats)
sample_std = np.std(test_stats, ddof=1)
sample_size = len(test_stats)

# Calculate the t-statistic and p-value
t_stat = (sample_mean - null_accuracy) / (sample_std)
p_value = 2 * stdtr(sample_size - 1, -np.abs(t_stat))

# Print the results
print("t-statistic:", t_stat)
print("p-value:", p_value)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
X_train_res, y_train_res = sm.fit_resample(X_train, y_train)
clf.fit(X_train_res, y_train_res)
# Make predictions on the test set
y_pred = clf.predict(X_test)
y_trainpred = clf.predict(X_train)
# Calculate the accuracy of the model
accuracyTrain = accuracy_score(y_train, y_trainpred)
accuracy = accuracy_score(y_test, y_pred)
print(accuracyTrain, "Train Accuracy")
print(accuracy, 'Test Accuracy')

label = list(df[category].unique())
label_t = []
for word in label:
    new_word = word + " True"
    label_t.append(new_word)
label_p = []
for word in label:
    new_word = word + " Pred"
    label_p.append(new_word)

# Compute the confusion matrix
conf_mat = confusion_matrix(y_test, y_pred, labels=label)
# Convert the matrix to a pandas DataFrame for easier manipulation
conf_df = pd.DataFrame(conf_mat, index=label_t, columns=label_p)
# Create a heatmap of the confusion matrix using seaborn
sb.heatmap(conf_df, annot=True, cmap='Blues', fmt='g')
# Generate the classification report
report = classification_report(y_test, y_pred)
print(report)

In [ ]:
# Define the number of folds
n_folds = 5

# Define the cross-validation method
kf = KFold(n_splits=n_folds, shuffle=True, random_state=42)
scores = cross_val_score(clf, X, y, cv=kf)
print(scores)
# Print the cross-validation scores
print(f"Cross-validation scores: {scores}")
print(f"Mean accuracy: {scores.mean():.3f}")
# Train the final model on the full dataset
clf.fit(X, y)

In [ ]:
# Define the number of folds
n_folds = 5

# Define the cross-validation method
kf = KFold(n_splits=n_folds, shuffle=True, random_state=42)

# Get the predicted target values using cross_val_predict()
y_pred = cross_val_predict(clf, X, y, cv=kf)

# Get the confusion matrix using confusion_matrix()
cm = confusion_matrix(y, y_pred)

# Print the confusion matrix
print("Confusion matrix:")
print(cm)

# Compute the confusion matrix
conf_mat = confusion_matrix(y, y_pred, labels=label)
# Convert the matrix to a pandas DataFrame for easier manipulation
conf_df = pd.DataFrame(conf_mat, index=label_t, columns=label_p)
# Create a heatmap of the confusion matrix using seaborn
sb.heatmap(conf_df, annot=True, cmap='Blues', fmt='g')
# Generate the classification report
report = classification_report(y, y_pred)
print(report)

In [ ]:
# Get the feature importance scores
importance_scores = clf.feature_importances_
feature_names = df.iloc[:, 15:].columns[:-1]
# Print the feature importance scores
for feature_name, importance_score in zip(feature_names, importance_scores):
    print(f'{feature_name}: {importance_score:.4f}')

In [ ]:
# Use the model to make predictions on the data
predictions = clf.predict(population_independent)

# Add the predictions as a new column to the DataFrame
popu = popu.assign(Family=predictions)
popu

# DBH Random Forest

In [ ]:
from sklearn.neighbors import RadiusNeighborsRegressor
from sklearn.ensemble import RandomForestRegressor
import seaborn as sns
import scipy.stats as stats
import numpy as np
import matplotlib.pyplot as plt

category = "DBH"
df = dfTrain
df = df.drop(df[df[category] == 0].index)
n_bootstrap = 1
# (467/1260)^2+(260/1260)^2+(231/1260)^2+(85/1260)^2+(78/1260)^2+(75/1260)^2+(64/1260)^2
null_accuracy = 0.2281
model = RandomForestRegressor(n_estimators=100, criterion='squared_error', max_depth=None, min_samples_split=2, min_samples_leaf=1, min_weight_fraction_leaf=0.0, max_features=1.0, max_leaf_nodes=None, min_impurity_decrease=0.0, bootstrap=True, oob_score=False, n_jobs=None, random_state=None, verbose=0, warm_start=False, ccp_alpha=0.0, max_samples=None)
X = df.iloc[:, 14:].astype(np.float64)
y = df[category]
X


In [ ]:
# bootstrap
# Bootstrap the data and calculate the test statistic on each bootstrap sample
test_stats = []
test_RMSE = []

for i in range(n_bootstrap):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
    model.fit(X_train, y_train)
    # Make predictions on the test set
    y_pred = model.predict(X_test)
    # Calculate the accuracy of the model
    test_r2 = model.score(X_test, y_test)
    test_rmse = mean_squared_error(y_test, y_pred, squared=False)
    test_stats.append(test_r2)
    test_RMSE.append(test_rmse)

# Calculate the mean
print("r2 value", np.mean(test_stats))

# Calculate the mean
mean = np.mean(test_RMSE)
# Calculate the standard deviation
std = np.std(test_RMSE)
# Confidence interval
n = len(test_RMSE)
confidence_level = 0.95
ci = t.interval(confidence_level, n - 1, loc=mean, scale=std)
print(f"Confidence interval: [{ci[0]:.3f}, {ci[1]:.3f}]")

# Print the results
print("Mean RMSE: {:.5f}".format(mean))
print("Standard deviation: {:.4f}".format(std))
plt.hist(test_RMSE, density=False)


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
model.fit(X_train, y_train)
# Make predictions on the test set
y_pred = model.predict(X_test)
# Calculate the accuracy of the model
# test_r2 = model.score(X_test, y_test)
test_rmse = mean_squared_error(y_test, y_pred, squared=False)
print("RMSE",test_rmse)



# plot the test_y and pred_y using jointplot with annotations
sns.set_style('whitegrid')
g = sns.jointplot(x=y_test, y=y_pred, kind='scatter', color='steelblue', height=8, ratio=5, marginal_ticks=True)
min_val = min(y_test.min(), y_pred.min())
max_val = max(y_test.max(), y_pred.max())
g.ax_joint.plot([min_val, max_val], [min_val, max_val], color='grey', linestyle='--')
g.ax_joint.set_xlabel('True DBH', fontsize=14)
g.ax_joint.set_ylabel('Predicted DBH', fontsize=14)
# g.ax_joint.set_title('Predicted vs Actual Values', fontsize=16)
plt.show()

In [ ]:
# Use the model to make predictions on the data
predictions = model.predict(population_independent)

# Add the predictions as a new column to the DataFrame
popu = popu.assign(DBH=predictions)
popu

In [ ]:
# Check if any value in the DataFrame is too large or infinite
if popu.isin([np.nan, np.inf, -np.inf]).any().any():
    print("There are infinite or too large values in the DataFrame.")
else:
    print("There are no infinite or too large values in the DataFrame.")

# Filter the DataFrame to show rows with infinite values
mask = popu.isin([np.inf, -np.inf])
filtered_df = popu.loc[mask.any(axis=1)]

# Print out the filtered DataFrame
print("The rows with infinite values in the DataFrame are:")
print(filtered_df)

# Exploration of Prediction Results

In [ ]:
# round the values in the column to the nearest integer
popu['DBH'] = popu['DBH'].round()

# calculate the minimum and maximum values of the column
min_value = int(popu['DBH'].min())
max_value = int(popu['DBH'].max())

# create bins with a width of 1
bins = range(min_value, max_value + 2, 1)

# plot the histogram
plt.hist(popu['DBH'], bins=bins)

# set the title and labels for the plot
plt.title('Distribution of DBH for all trees')
plt.xlabel('DBH')
plt.ylabel('Frequency')

# show the plot
plt.show()

In [ ]:
# count the frequency of each category in the column
category_counts = popu['Functional'].value_counts()

# create a bar chart of the category counts
category_counts.plot.bar()

# create a bar chart of the category counts
ax = category_counts.plot.bar()

# set the title and labels for the plot
plt.title('Bar Chart of broadleaf deciduous, hedge, and coniferous evergreen')
plt.xlabel('Functional')
plt.ylabel('Frequency')

# add text labels for the counts on each bar
for i, v in enumerate(category_counts.values):
    ax.text(i, v + 0.2, str(v), ha='center')

# show the plot
plt.show()

In [ ]:
# count the frequency of each category in the column
category_counts = popu['Family'].value_counts()

# create a bar chart of the category counts
category_counts.plot.bar()

# create a bar chart of the category counts
ax = category_counts.plot.bar()

# set the title and labels for the plot
plt.title('Bar Chart of Family Classification')
plt.xlabel('Family')
plt.ylabel('Frequency')

# add text labels for the counts on each bar
for i, v in enumerate(category_counts.values):
    ax.text(i, v + 0.2, str(v), ha='center')

# show the plot
plt.show()

In [ ]:
# count the frequency of each category in the column
category_counts = popu['Condition'].value_counts()

# create a bar chart of the category counts
category_counts.plot.bar()

# create a bar chart of the category counts
ax = category_counts.plot.bar()

# set the title and labels for the plot
plt.title('Bar Chart of Condition Classification')
plt.xlabel('Condition')
plt.ylabel('Frequency')

# add text labels for the counts on each bar
for i, v in enumerate(category_counts.values):
    ax.text(i, v + 0.2, str(v), ha='center')

# show the plot
plt.show()

In [ ]:

# export the dataframe to a CSV file
popu.to_csv('/content/drive/MyDrive/XihanYAO_GISDataShare/230318_Prediction.csv', index=False)